# Flares de AU Mic no SPARC4 — temperatura e energia

A fotometria já vem pronta nos produtos `*_lc.fits` do pipeline: magnitude de
abertura de **todas** as fontes do campo, em todos os instantes, nos quatro
canais. Este notebook não refaz redução — ele monta a curva **diferencial** nas
quatro bandas, identifica as flares e mede **T(t)**, **área** e **energia
bolométrica** pelo método de Heinzel+2026.

## Organização

| célula | o que faz |
|:--:|---|
| 1 | **configuração** — tudo que se ajusta está aqui |
| 2 | leitura dos produtos do pipeline |
| 3 | curva diferencial nas 4 bandas, grade comum, quiescente → `C(t)` |
| 4 | bandas SPARC4: função resposta real, do CSV |
| 5 | modelo de flare (Heinzel+2026): Eqs. 6–9 |
| 6 | detecção das flares |
| 7 | `analisa_flare()` — T, área e energia + incertezas Monte Carlo |
| 8 | figuras |
| 9 | a noite, de ponta a ponta |
| 10 | tabela das flares |
| 11 | robustez: o quanto o resultado depende das escolhas |
| 12 | arquivos gravados |

## Os quatro canais não compartilham a grade de tempo

Cada canal tem cadência própria (em 2026-08-15: 4.1 s no `g`, 1.3 s nos outros).
Como o método é **multibanda ponto a ponto**, tudo é rebinado numa **grade comum**
de `BIN_S` segundos (célula 3) antes de qualquer ajuste — é essa grade, e não a
cadência do instrumento, que define a resolução temporal de `T(t)`.

## O que é medida e o que não é

A **temperatura de pico multibanda** e a **energia** dela derivada são a medida:
saem do formato da SED nas quatro bandas, sem assumir T. Já a `T(t)` **fora do
pico** é ruidosa — longe do máximo a amplitude cai para o nível do ruído em
`i`/`z` e a cor deixa de restringir T. Por isso a energia é dada em três
versões (célula 7):

- **IIa** — área fixa da amplitude de pico, com `T_PICO_IIA` **assumido**;
- **IIb** — o mesmo, mas com o T de pico **medido** na SED (é o número a citar);
- **MB** — T e área livres em cada instante; serve de diagnóstico e de limite
  superior, porque o ruído em T entra na energia como `T⁴`. Fora do pico o
  ajuste só roda onde alguma banda passa de `NSIG_MB`: sem essa porta, os
  instantes de baixa razão sinal-ruído escorregam para o teto de `T_LIM` com
  área minúscula e passam a dominar a integral da energia.

A área sai sempre **deprojetada** por `MU`, e a energia tem o piso quiescente
`σT_star⁴` subtraído.

In [30]:
# ═══ 1 · configuração ═══════════════════════════════════════════════════════
# Tudo que se ajusta está nesta célula; as seguintes só consomem estes valores.
import os

BASE    = r"C:/Users/paola/Desktop/Doutorado/Codigos-doc/Flares-/novo_homepage/Sparc4-data"
DATADIR = os.path.join(BASE, "data")
OUTDIR  = os.path.join(BASE, "resultados_flares")
RESPOSTA = os.path.join(DATADIR, "sparc4_spectral_response.csv")
os.makedirs(OUTDIR, exist_ok=True)

# ── instrumento ─────────────────────────────────────────────────────────────
CANAIS = [1, 2, 3, 4]
BANDAS = {1: "g", 2: "r", 3: "i", 4: "z"}
CORES  = {"g": "#3b6ea5", "r": "#3f8f4f", "i": "#c06a2a", "z": "#8b3a5a"}

BEAM   = "S+N"     # soma dos dois feixes: cancela a modulação da lâmina
APER   = 8         # raio da abertura, em pixels (tem que existir no produto)
NCOMPS = 3         # nº de comparações: as mais brilhantes depois do alvo
COMPS  = {}        # força as comparações de um canal, p.ex. {1: [1, 3, 2]}

# ── noites ──────────────────────────────────────────────────────────────────
NOITES = ["20260815"]      # [] = todas as noites com os quatro canais

# ── estrela: AU Mic ─────────────────────────────────────────────────────────
T_STAR = 3700.0    # K        (Plavchan+2020)
R_STAR = 0.75      # R_sol    (Plavchan+2020)
MU     = 1.0       # cos(theta) da região da flare; 1 = no centro do disco.
                   # A área projetada medida é dividida por MU para deprojetar,
                   # então MU=1 dá o LIMITE INFERIOR da área e da energia.

# ── grade comum e quiescente ────────────────────────────────────────────────
BIN_S     = 15.0   # grade comum das 4 bandas, em segundos
MIN_EXP   = 2      # nº mínimo de exposições por bin, por banda
QUIESC    = "mediana"   # "mediana" (móvel, local) ou "poly" (ordem POLY_ORD)
JANELA_Q  = 20.0   # largura da mediana móvel do quiescente, em minutos
POLY_ORD  = 2      # ordem do polinômio, se QUIESC == "poly"
NSIG_Q    = 3.0    # acima disto o ponto sai do quiescente (é flare)
NSIG_BAIXO = 6.0   # abaixo disto o ponto sai como nuvem / guiagem ruim
NITER_Q   = 4      # iterações do quiescente

# ── detecção de flares ──────────────────────────────────────────────────────
BANDA_DET  = "g"   # banda onde a flare é procurada (a de maior contraste)
NSIG_INI   = 3.0   # limiar de entrada, em sigma do quiescente
NSIG_FIM   = 1.0   # a flare só termina quando o fluxo cai abaixo disto
NSIG_PICO  = 5.0   # o pico tem que passar disto, senão não é flare
NPTS_MIN   = 3     # nº mínimo de bins consecutivos acima de NSIG_INI
NBANDAS_MIN = 2    # em quantas bandas o pico precisa aparecer (corta raio cósmico)
JUNTA_MIN  = 1.0   # flares separadas por menos que isto viram uma só, em minutos
PAD_MIN    = 2.0   # margem de quiescente de cada lado da janela, em minutos

# ── medida ──────────────────────────────────────────────────────────────────
T_PICO_IIA = 10000.0        # K, o T assumido do método IIa (só ele usa)
T_LIM      = (3800., 50000.)  # limites do T no ajuste multibanda
NSIG_MB    = 3.0            # o MB só ajusta T onde alguma banda passa disto;
                            # abaixo, a cor não restringe T e o ajuste foge
N_PICO     = 3              # nº de bins em torno do máximo que definem o pico
NMC        = 300            # realizações Monte Carlo das incertezas (0 desliga)
SEMENTE    = 42

print(f"dados:      {DATADIR}")
print(f"resposta:   {RESPOSTA}")
print(f"resultados: {OUTDIR}")
print(f"AU Mic:     T* = {T_STAR:.0f} K | R* = {R_STAR} R_sol | mu = {MU}")
print(f"grade comum de {BIN_S:.0f} s | flares procuradas na banda {BANDA_DET}")

dados:      C:/Users/paola/Desktop/Doutorado/Codigos-doc/Flares-/novo_homepage/Sparc4-data\data
resposta:   C:/Users/paola/Desktop/Doutorado/Codigos-doc/Flares-/novo_homepage/Sparc4-data\data\sparc4_spectral_response.csv
resultados: C:/Users/paola/Desktop/Doutorado/Codigos-doc/Flares-/novo_homepage/Sparc4-data\resultados_flares
AU Mic:     T* = 3700 K | R* = 0.75 R_sol | mu = 1.0
grade comum de 15 s | flares procuradas na banda g


In [31]:
# ═══ 2 · leitura dos produtos do pipeline ═══════════════════════════════════
# O produto *_lc.fits traz uma extensão CATALOG_PHOT_APxxx por raio de abertura,
# com uma linha por (fonte, instante). Aqui isso vira uma matriz (fonte, tempo).
import glob
import numpy as np
from astropy.io import fits
from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u


def caminho_lc(noite, ch):
    """Produto de fotometria da noite/canal (o do feixe BEAM)."""
    padrao = os.path.join(DATADIR, noite, f"*_s4c{ch}_*_{BEAM}_lc.fits")
    arqs = sorted(glob.glob(padrao))
    if not arqs:
        raise FileNotFoundError(f"nenhum produto casa com {padrao}")
    return arqs[0]


def cabecalho(noite, ch=1):
    """Cabeçalho primário — TSTART/TSTOP, coordenadas, sítio."""
    return fits.getheader(caminho_lc(noite, ch))


def le_fotometria(noite, ch, aper=APER):
    """Fotometria pronta do pipeline, já organizada.

    Devolve (hdr, t_utc, idx, MAG, EMAG), com MAG e EMAG na forma
    (fonte, tempo) e NaN onde o pipeline levantou FLAG.
    """
    with fits.open(caminho_lc(noite, ch)) as h:
        exts = [e for e in h[1:] if e.header.get("APRADIUS") == aper]
        if not exts:
            disp = sorted({e.header.get("APRADIUS") for e in h[1:]})
            raise KeyError(f"abertura {aper} px não existe; há {disp}")
        d, hdr = exts[0].data, h[0].header
        src  = np.asarray(d["SRCINDEX"], float).astype(int)
        t    = np.asarray(d["TIME"], float)
        mag  = np.asarray(d["MAG"], float)
        emag = np.asarray(d["EMAG"], float)
        flag = np.asarray(d["FLAG"]) if "FLAG" in d.columns.names else None

    idx, t_utc = np.unique(src), np.unique(t)
    lin, col = np.searchsorted(idx, src), np.searchsorted(t_utc, t)
    M = np.full((idx.size, t_utc.size), np.nan)
    E = np.full_like(M, np.nan)
    M[lin, col], E[lin, col] = mag, emag
    if flag is not None:
        ruim = np.zeros(M.shape, bool)
        ruim[lin, col] = np.asarray(flag) != 0
        M[ruim] = np.nan
    return hdr, t_utc, idx, M, E


def massa_de_ar(t_utc, hdr):
    """sec(z) do alvo, do sítio e das coordenadas gravadas no cabeçalho."""
    sitio = EarthLocation(lat=hdr["OBSLAT"] * u.deg, lon=hdr["OBSLONG"] * u.deg,
                          height=hdr["OBSALT"] * u.m)
    alvo  = SkyCoord(hdr["RA"] * u.deg, hdr["DEC"] * u.deg, frame="icrs")
    tempo = Time(t_utc, format="jd", scale="utc")
    return alvo.transform_to(AltAz(obstime=tempo, location=sitio)).secz.value


def rotulo_noite(noite):
    return f"{noite[:4]}-{noite[4:6]}-{noite[6:]}"


def noites_disponiveis():
    """Noites com produto nos quatro canais."""
    saida = []
    for d in sorted(glob.glob(os.path.join(DATADIR, "*", ""))):
        noite = os.path.basename(os.path.normpath(d))
        if all(glob.glob(os.path.join(d, f"*_s4c{ch}_*_{BEAM}_lc.fits"))
               for ch in CANAIS):
            saida.append(noite)
    return saida


NOITES_USO = NOITES or noites_disponiveis()
print("noites com os quatro canais:", ", ".join(noites_disponiveis()))
print("noites deste notebook:      ", ", ".join(rotulo_noite(n) for n in NOITES_USO))

noites com os quatro canais: 20260702, 20260713, 20260714, 20260719, 20260731, 20260803, 20260805, 20260815, 20260818, 20260822
noites deste notebook:       2026-08-15


In [32]:
# ═══ 3 · curva diferencial, grade comum e quiescente ════════════════════════
# alvo / soma das comparações em cada canal, tudo rebinado numa grade única, e
# o nível quiescente ajustado com as flares mascaradas. O produto é C(t) = f/q.


def curva_diferencial(noite, ch, aper=APER, ncomps=NCOMPS, comps=None):
    """Razão de fluxos alvo / Σ comparações, a partir da fotometria do pipeline.

    O alvo é a fonte mais brilhante do campo e as comparações são as `ncomps`
    seguintes, a não ser que `comps` traga índices explícitos. Ao contrário do
    notebook de trânsito, aqui NENHUM corte de outlier é aplicado: a flare é,
    por definição, um outlier positivo.
    """
    hdr, t_utc, idx, M, E = le_fotometria(noite, ch, aper)

    ordem = idx[np.argsort(np.nanmedian(M, axis=1))]
    alvo  = int(ordem[0])
    comps = [int(c) for c in (comps if comps else ordem[1:1 + ncomps])]
    ia = int(np.where(idx == alvo)[0][0])
    ic = [int(np.where(idx == c)[0][0]) for c in comps]

    with np.errstate(invalid="ignore", divide="ignore"):
        f_alvo = 10 ** (-0.4 * M[ia])
        f_comp = np.nansum(10 ** (-0.4 * M[ic]), axis=0)
        razao  = f_alvo / f_comp
        e_comp = np.sqrt(np.nansum((10 ** (-0.4 * M[ic]) * E[ic]) ** 2, axis=0))
        e_raz  = razao * 0.4 * np.log(10) * np.hypot(
            E[ia], e_comp / np.where(f_comp > 0, f_comp, np.nan))

    ok = np.isfinite(razao) & np.isfinite(e_raz)
    t_tdb = Time(t_utc[ok], format="jd", scale="utc").tdb.jd   # BJD_UTC -> BJD_TDB
    return dict(noite=noite, canal=ch, banda=BANDAS[ch], alvo=alvo, comps=comps,
                cadencia_s=float(np.median(np.diff(t_utc)) * 86400),
                t=t_tdb, f=razao[ok] / np.nanmedian(razao[ok]), ef=e_raz[ok],
                X=massa_de_ar(t_utc[ok], hdr))


def rebina(t, f, bordas, min_exp=MIN_EXP):
    """Média em bins fixos; o erro do bin é a dispersão interna dividida por √n.

    Bins com menos de `min_exp` exposições saem como NaN — é o que garante que
    as quatro bandas caiam exatamente nos mesmos instantes.
    """
    n_bins = bordas.size - 1
    k = np.clip(np.searchsorted(bordas, t, "right") - 1, 0, n_bins - 1)
    n  = np.bincount(k, minlength=n_bins).astype(float)
    s1 = np.bincount(k, weights=f, minlength=n_bins)
    s2 = np.bincount(k, weights=f * f, minlength=n_bins)
    with np.errstate(invalid="ignore", divide="ignore"):
        media = s1 / n
        var   = (s2 - n * media ** 2) / (n - 1)
        erro  = np.sqrt(np.maximum(var, 0.0) / n)
    fora = n < min_exp
    media[fora], erro[fora], n[fora] = np.nan, np.nan, 0
    return media, erro, n


def quiescente(t, f, modo=QUIESC, janela_min=JANELA_Q, ordem=POLY_ORD,
               nsig=NSIG_Q, nsig_baixo=NSIG_BAIXO, niter=NITER_Q):
    """Nível quiescente com as flares mascaradas, iterativamente.

    Cada passada ajusta o quiescente só nos pontos ainda considerados calmos,
    mede o sigma do resíduo e remarca o que sobe acima de `nsig` (flare) ou
    desce abaixo de `nsig_baixo` (nuvem, guiagem). O corte é ASSIMÉTRICO de
    propósito: subida é sinal, descida é problema.

    A mediana móvel corre sobre os pontos SOBREVIVENTES, não sobre a grade: a
    janela é de `janela_min` de dados calmos, e o resultado é interpolado de
    volta na grade cheia. Uma mediana sobre a grade, com a janela fixa em
    índice, teria dentro dela a flare inteira quando `janela_min` não fosse
    muito maior que a duração da flare — e devolveria um quiescente puxado para
    cima justamente onde ele precisa estar certo.
    """
    tm  = (t - t[0]) * 1440
    bom = np.isfinite(f)
    q = np.full_like(f, np.nan)
    sig = np.nan
    for _ in range(niter):
        if modo == "poly":
            c = np.polyfit(tm[bom], f[bom], ordem)
            q = np.polyval(c, tm)
        else:
            ig = np.flatnonzero(bom)
            yg = f[ig]
            w = max(int(round(janela_min / np.median(np.diff(tm)))) | 1, 5)
            w = min(w, yg.size if yg.size % 2 else yg.size - 1)
            pad = w // 2
            ext = np.concatenate([yg[:pad][::-1], yg, yg[-pad:][::-1]])
                "caminho_curvas = os.path.join(OUTDIR, \"aumic_flares_20260815_F01_curvas.csv\")",
                "dados = pd.read_csv(caminho_curvas)",
        sig = 1.4826 * np.nanmedian(np.abs(r[bom] - np.nanmedian(r[bom])))
        bom = np.isfinite(f) & (r < nsig * sig) & (r > -nsig_baixo * sig)
    return q, float(sig), bom
                "mu = MU"

def prepara_noite(noite, bin_s=BIN_S, aper=APER, ncomps=NCOMPS, canais=None,
                  modo_q=QUIESC, janela_q=JANELA_Q):
    """Da fotometria bruta a C(t) nas quatro bandas, na mesma grade de tempo."""
    curvas = {BANDAS[ch]: curva_diferencial(noite, ch, aper, ncomps, COMPS.get(ch))
              for ch in (canais or CANAIS)}
    nomes = list(curvas)

    t0 = min(c["t"].min() for c in curvas.values())
    t1 = max(c["t"].max() for c in curvas.values())
    bordas = np.arange(t0, t1 + bin_s / 86400, bin_s / 86400)
    t = 0.5 * (bordas[:-1] + bordas[1:])

    F, EF, N, Q, C, SIG, CALMO, RUIM = {}, {}, {}, {}, {}, {}, {}, {}
    for n in nomes:
        c = curvas[n]
        f, ef, cont = rebina(c["t"], c["f"], bordas)
        f = f / np.nanmedian(f)
        q, sig, bom = quiescente(t, f, modo=modo_q, janela_min=janela_q)
        # piso de erro: o ruído branco ponto a ponto medido na própria curva
        piso = np.nanstd(np.diff(f[np.isfinite(f)])) / np.sqrt(2)
        F[n], EF[n], N[n] = f, np.fmax(np.nan_to_num(ef, nan=piso), piso), cont
        Q[n], SIG[n] = q, sig
        C[n] = f / q
        CALMO[n] = bom
        RUIM[n] = np.isfinite(f) & (f - q < -NSIG_BAIXO * sig)

    completo = np.all([np.isfinite(F[n]) & ~RUIM[n] for n in nomes], axis=0)
    X = np.interp(t, curvas[nomes[0]]["t"], curvas[nomes[0]]["X"])
    return dict(noite=noite, bandas=nomes, curvas=curvas, bin_s=bin_s,
                t=t, X=X, f=F, ef=EF, n=N, q=Q, C=C, sig=SIG, ruim=RUIM,
                calmo=CALMO,
                completo=completo, modo_q=modo_q, janela_q=janela_q,
                aper=aper, ncomps=ncomps,
                dur_h=float((t[-1] - t[0]) * 24))


def resumo_noite(d):
    print(f"── {rotulo_noite(d['noite'])} " + "─" * 52)
    print(f"{d['dur_h']:.2f} h | grade de {d['bin_s']:.0f} s -> {d['t'].size} bins "
          f"({d['completo'].sum()} com as {len(d['bandas'])} bandas boas) | "
          f"quiescente: {d['modo_q']}")
    for n in d["bandas"]:
        cad = d["curvas"][n]["cadencia_s"]
        print(f"  {n}: cadência {cad:5.2f} s | σ quiescente {d['sig'][n]*1e3:5.2f} ppt "
              f"| C máx {np.nanmax(d['C'][n]):.4f} | "
              f"comparações {d['curvas'][n]['comps']}")

IndentationError: unexpected indent (725436558.py, line 89)

In [4]:
# ═══ 4 · bandas SPARC4: a função resposta real ══════════════════════════════
# S_lambda dos quatro canais, lido do CSV do instrumento. É S_lambda que entra
# nas Eqs. (6)-(7) de Heinzel+2026 — trocá-lo por um top-hat muda a cor prevista
# e, com ela, o T ajustado.

H, C_LIGHT, KB = 6.62607015e-34, 2.99792458e8, 1.380649e-23
SIGMA_SB = 5.670374419e-5      # erg s^-1 cm^-2 K^-4
R_SUN_CM = 6.957e10


def _planck(lam, T):
    """B_lambda(T) para um vetor de lambda [m] e um vetor de T [K]."""
    T = np.atleast_1d(np.asarray(T, float))[:, None]
    x = np.clip(H * C_LIGHT / (lam[None, :] * KB * T), 1e-10, 700.0)
    return 2.0 * H * C_LIGHT ** 2 / (lam[None, :] ** 5 * np.expm1(x))


class Banda:
    """Banda fotométrica com função resposta S_lambda real.

    A integral I(T) = ∫ S_lambda B_lambda(T) dlambda é tabelada uma vez numa
    grade de T e depois só interpolada — é o que torna viável rodá-la para cada
    instante de cada realização Monte Carlo. A inversão T(I) é a mesma tabela
    lida ao contrário (I é monotônica em T), o que dispensa um brentq por ponto.
    """

    def __init__(self, nome, lam_m, S, T_grid=None):
        i = np.argsort(lam_m)
        self.nome, self.lam, self.S = nome, np.asarray(lam_m)[i], np.asarray(S)[i]
        ok = self.S > 1e-4 * self.S.max()
        self.lam, self.S = self.lam[ok], self.S[ok]
        self.T_grid = (np.logspace(np.log10(1500.), np.log10(60000.), 4000)
                       if T_grid is None else T_grid)
        self.I_grid = np.trapezoid(_planck(self.lam, self.T_grid) * self.S[None, :],
                                   self.lam, axis=1)

    @classmethod
    def tophat(cls, nome, lam_min, lam_max, n=400):
        lam = np.linspace(lam_min, lam_max, n)
        return cls(nome, lam, np.ones_like(lam))

    @classmethod
    def de_csv(cls, caminho, unidade="nm", escala=0.01):
        """Todas as bandas de um CSV `lambda, S_g, S_r, S_i, S_z`.

        `escala=0.01` converte transmissão em % para fração. Devolve um dict
        {nome: Banda}, com o nome tirado do cabeçalho ("Channel g (%)" -> "g").
        """
        import pandas as pd
        d = pd.read_csv(caminho)
        fator = {"m": 1.0, "nm": 1e-9, "A": 1e-10, "um": 1e-6}[unidade]
        lam = d.iloc[:, 0].to_numpy(float) * fator
        saida = {}
        for col in d.columns[1:]:
            partes = col.split()
            nome = partes[1] if partes[0].lower() == "channel" else partes[0]
            saida[nome] = cls(nome, lam, d[col].to_numpy(float) * escala)
        return saida

    def lambda_efetivo(self):
        return float(np.trapezoid(self.lam * self.S, self.lam)
                     / np.trapezoid(self.S, self.lam))

    def I(self, T):                 # ∫ S_lam B_lam(T) dlam   (tabelado)
        return np.interp(T, self.T_grid, self.I_grid)

    def T_de_I(self, I):            # inversão monotônica
        return np.interp(I, self.I_grid, self.T_grid, left=np.nan, right=np.nan)


BANDAS_SPARC4 = Banda.de_csv(RESPOSTA, unidade="nm")

print(f"função resposta: {os.path.basename(RESPOSTA)}")
for n, b in BANDAS_SPARC4.items():
    print(f"  {n}: {b.lam.min()*1e9:6.1f}-{b.lam.max()*1e9:6.1f} nm | "
          f"lambda_ef = {b.lambda_efetivo()*1e9:5.1f} nm | "
          f"S_max = {b.S.max():.3f} | {b.lam.size} pontos")

função resposta: sparc4_spectral_response.csv
  g:  350.0- 683.3 nm | lambda_ef = 457.2 nm | S_max = 0.821 | 45 pontos
  r:  448.5-1001.5 nm | lambda_ef = 613.6 nm | S_max = 0.928 | 74 pontos
  i:  456.1-1062.1 nm | lambda_ef = 753.3 nm | S_max = 0.774 | 81 pontos
  z:  615.2-1100.0 nm | lambda_ef = 894.3 nm | S_max = 0.438 | 65 pontos


In [19]:
# ═══ 5 · modelo de flare — Heinzel+2026 ═════════════════════════════════════
# A flare é um corpo negro de temperatura T e área A somado ao disco quiescente:
#
#   Eq. (6)  dC_max -> A, fixa, a partir da amplitude de pico e de um T assumido
#   Eq. (7)  A fixa -> T(t), instante a instante, de UMA banda
#   Eq. (8)  L(t) = sigma T(t)^4 A          (com o piso quiescente subtraído)
#   Eq. (9)  E    = integral de L dt
#
# `ajuste_multibanda` é a variante SPARC4: com quatro bandas, T e A saem JUNTOS
# de cada instante e não é preciso assumir T de pico nenhum.
from scipy.optimize import minimize_scalar


class FlareHeinzel:
    def __init__(self, T_star, R_star, bandas, mu=1.0):
        self.T_star, self.R_star, self.mu = T_star, R_star, mu 
        self.A_star = np.pi * (R_star * R_SUN_CM) ** 2   # OK
        self.bandas = {b.nome: b for b in bandas} # OK
        self.I_star = {n: float(b.I(T_star)) for n, b in self.bandas.items()} # OK

    # ---- Eq. (6): área projetada, fixa, a partir da amplitude de pico -------
    def area_pico(self, dC_max, T_pico, banda):
        b = self.bandas[banda]
        den = b.I(T_pico) - self.I_star[banda]
        if den <= 0:
            raise ValueError("T_pico deve ser > T_star.")
        A_proj = dC_max * self.I_star[banda] / den * self.A_star
        return A_proj, A_proj / self.mu          # (projetada, deprojetada)

    # ---- Eq. (7): T_flare(t) com área fixa ---------------------------------
    def temperatura(self, C, A_proj, banda, dC_min=0.0):
        dC = np.asarray(C, float) - 1.0
        Is = self.I_star[banda]
        I_alvo = np.where(dC > dC_min, dC * self.A_star / A_proj * Is + Is, Is)
        T = self.bandas[banda].T_de_I(I_alvo)
        return np.where(np.isnan(T), self.T_star, np.maximum(T, self.T_star))

    # ---- Eqs. (8)-(9): energia bolométrica ---------------------------------
    def energia(self, t_dias, T, A_deproj, subtrair_piso=True):
        t = np.asarray(t_dias, float) * 86400.0
        T4 = np.asarray(T, float) ** 4
        if subtrair_piso:
            T4 = np.clip(T4 - self.T_star ** 4, 0.0, None)  # remove o quiescente
        L = SIGMA_SB * T4 * np.asarray(A_deproj, float)
        return float(np.trapezoid(L, t)), L

    # ---- multibanda: (T, A) por época, sem assumir T_pico ------------------
    def ajuste_multibanda(self, fluxos, sigmas=None, T_lim=T_LIM,
                          nsig_min=NSIG_MB):
        """T e área em cada instante, das quatro bandas.

        Para um T de teste a razão de áreas entra LINEARMENTE em dC, então ela
        sai em forma fechada e sobra só uma busca 1-D em T. `chi2` é a soma dos
        resíduos pesados, com (n_bandas - 2) graus de liberdade.

        `nsig_min` é o que impede o ajuste de fugir: onde NENHUMA banda passa
        desse nível de significância, a cor não restringe T — o mínimo escorrega
        para o teto de `T_lim` com área minúscula, e como a energia vai com T⁴
        um punhado desses instantes domina a integral. Esses instantes voltam
        com T = T_star e área zero, isto é, sem contribuir energia nenhuma.
        """
        nomes = list(fluxos)
        dC = np.array([np.asarray(fluxos[n], float) - 1.0 for n in nomes])
        w = (np.ones_like(dC) if sigmas is None
             else 1.0 / np.array([np.asarray(sigmas[n], float) for n in nomes]) ** 2)
        Is = np.array([self.I_star[n] for n in nomes])
        Ts, As, chi2 = [], [], []
        for k in range(dC.shape[1]):
            y, wk = dC[:, k], w[:, k]
            fraco = (sigmas is not None
                     and np.max(y * np.sqrt(wk)) < nsig_min)
            if not np.all(np.isfinite(y)) or not np.any(y > 0) or fraco:
                Ts.append(self.T_star); As.append(0.0); chi2.append(np.nan)
                continue

            def custo(T):
                g = np.array([self.bandas[n].I(T) for n in nomes]) / Is - 1.0
                r = max(np.sum(wk * y * g) / np.sum(wk * g * g), 0.0)
                return float(np.sum(wk * (y - r * g) ** 2)), r

            res = minimize_scalar(lambda T: custo(T)[0], bounds=T_lim, method="bounded")
            c, r = custo(res.x)
            Ts.append(float(res.x)); As.append(r * self.A_star / self.mu); chi2.append(c)
        return np.array(Ts), np.array(As), np.array(chi2)

    # ---- um único T para a SED de um trecho (o pico) ----------------------
    def temperatura_sed(self, dC, sigmas=None, T_lim=T_LIM, nsig_min=NSIG_MB):
        """T e área de UMA SED: os dC das bandas num instante ou trecho médio.

        É o `ajuste_multibanda` com uma coluna só, e é daqui que sai o T de pico
        MEDIDO que alimenta o método IIb. A mesma porta `nsig_min` vale aqui: se
        nem no pico a SED for significativa, volta T = T_star e o método IIb sai
        como NaN em vez de sair errado.
        """
        fl = {n: np.array([1.0 + v]) for n, v in dC.items()}
        sg = None if sigmas is None else {n: np.array([v]) for n, v in sigmas.items()}
        T, A, c = self.ajuste_multibanda(fl, sg, T_lim, nsig_min)
        return float(T[0]), float(A[0]), float(c[0])


modelo = FlareHeinzel(T_STAR, R_STAR, list(BANDAS_SPARC4.values()), mu=MU)

print(f"A_star (disco projetado) = {modelo.A_star:.4e} cm2")
print("contraste de uma area de 1e18 cm2 a 10 000 K, por banda:")
for n, b in modelo.bandas.items():
    dC = 1e18 / modelo.A_star * (float(b.I(1e4)) / modelo.I_star[n] - 1.0)
    print(f"  {n}: dC = {dC*1e3:6.2f} ppt")

A_star (disco projetado) = 8.5530e+21 cm2
contraste de uma area de 1e18 cm2 a 10 000 K, por banda:
  g: dC =  26.02 ppt
  r: dC =   7.11 ppt
  i: dC =   3.50 ppt
  z: dC =   2.17 ppt


In [20]:
# ═══ 6 · detecção das flares ════════════════════════════════════════════════
# Critério clássico, sobre o resíduo em sigma do quiescente na banda BANDA_DET:
# NPTS_MIN bins consecutivos acima de NSIG_INI, com o pico passando de
# NSIG_PICO. A janela é então ESTENDIDA até o fluxo cair abaixo de NSIG_FIM,
# para não cortar o decaimento — que é onde mora boa parte da energia.
import pandas as pd


def _trechos(mascara):
    """Índices (início, fim+1) de cada trecho contíguo verdadeiro."""
    m = np.concatenate([[0], mascara.astype(np.int8), [0]])
    d = np.diff(m)
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)))


def detecta_flares(d, banda=BANDA_DET, nsig_ini=NSIG_INI, nsig_fim=NSIG_FIM,
                   nsig_pico=NSIG_PICO, npts_min=NPTS_MIN, junta_min=JUNTA_MIN,
                   pad_min=PAD_MIN, nbandas_min=NBANDAS_MIN):
    t, dt_min = d["t"], d["bin_s"] / 60.0
    f, q, s = d["f"][banda], d["q"][banda], d["sig"][banda]
    ok  = np.isfinite(f) & ~d["ruim"][banda]
    res = np.where(ok, (f - q) / s, 0.0)
    res_cru = np.where(np.isfinite(f), (f - q) / s, np.nan)   # sem máscara

    # 1. gatilho, 2. extensão até o fluxo voltar ao quiescente
    cand = []
    for a, b in _trechos(res > nsig_ini):
        if b - a < npts_min:
            continue
        while a > 0 and res[a - 1] > nsig_fim:
            a -= 1
        while b < t.size and res[b] > nsig_fim:
            b += 1
        cand.append([a, b])

    # 3. flares que quase se encostam são uma só (pico complexo)
    juntas = []
    for a, b in sorted(cand):
        if juntas and (t[a] - t[juntas[-1][1] - 1]) * 1440 < junta_min:
            juntas[-1][1] = max(juntas[-1][1], b)
        else:
            juntas.append([a, b])

    # 4. peneira: pico significativo e presente em mais de uma banda
    pad = max(int(round(pad_min / dt_min)), 1)
    flares = []
    for a, b in juntas:
        if res[a:b].max() < nsig_pico:
            continue
        ipico = a + int(np.nanargmax(np.where(ok[a:b], f[a:b] - q[a:b], -np.inf)))
        nb = sum(((d["f"][n][ipico] - d["q"][n][ipico]) / d["sig"][n] > 3.0)
                 for n in d["bandas"] if np.isfinite(d["f"][n][ipico]))
        if nb < nbandas_min:
            continue
        j0, j1 = max(a - pad, 0), min(b + pad, t.size)
        C = d["C"][banda]
        dur = (t[b - 1] - t[a]) * 1440
        # o quiescente sob a flare é INTERPOLADO: só o que sobra de cada lado o
        # sustenta. Poucos bins antes, ou uma base que já não casa com os dados
        # ali, é a maior fonte de erro na energia — daí estes dois números.
        pre, pos = res_cru[j0:a], res_cru[b:j1]
        base_pre = float(np.nanmean(pre)) if np.isfinite(pre).any() else np.nan
        base_pos = float(np.nanmean(pos)) if np.isfinite(pos).any() else np.nan
        ed = float(np.trapezoid(np.nan_to_num(C[a:b] - 1.0), t[a:b]) * 86400)
        flares.append(dict(
            noite=d["noite"], banda_det=banda,
            i0=int(a), i1=int(b), j0=int(j0), j1=int(j1), ipico=int(ipico),
            t_ini=float(t[a]), t_fim=float(t[b - 1]), t_pico=float(t[ipico]),
            dur_min=float(dur), pico_sig=float(res[a:b].max()),
            ed_s=ed, n_bandas=int(nb), n_bins=int(b - a),
            n_pre=int(np.isfinite(pre).sum()), n_pos=int(np.isfinite(pos).sum()),
            base_pre_sig=base_pre, base_pos_sig=base_pos,
            amp={n: float(d["C"][n][ipico] - 1.0) for n in d["bandas"]},
            amp_sig={n: float((d["f"][n][ipico] - d["q"][n][ipico]) / d["sig"][n])
                     for n in d["bandas"]}))

    for k, fl in enumerate(flares, start=1):
        fl["id"] = f"{d['noite']}_F{k:02d}"
    return flares


def tabela_deteccao(flares):
    linhas = []
    for fl in flares:
        linha = {"flare": fl["id"],
                 "início (UTC)": Time(fl["t_ini"], format="jd").isot[11:19],
                 "dur (min)": fl["dur_min"], "bins": fl["n_bins"],
                 "pico (σ)": fl["pico_sig"], "ED (s)": fl["ed_s"]}
        linha |= {f"amp {n} (ppt)": v * 1e3 for n, v in fl["amp"].items()}
        linhas.append(linha)
    return pd.DataFrame(linhas)

In [15]:
# ═══ 7 · a medida: temperatura, área e energia ══════════════════════════════
# Três caminhos sobre a MESMA janela, para separar o que é dado do que é
# suposição:
#
#   IIa  área da amplitude de pico com T_PICO_IIA ASSUMIDO  -> T(t) de uma banda
#   IIb  idem, mas com o T de pico MEDIDO na SED das 4 bandas   <- a medida
#   MB   T e área livres em CADA instante (Heinzel multibanda)  -> diagnóstico
#
# As incertezas saem de Monte Carlo: cada realização reperturba as 4 curvas
# pelos seus erros e refaz tudo, inclusive a escolha do pico.


def _sigma_C(d, n, fatia):
    """Erro de C(t) numa banda: o do bin, com piso no espalhamento quiescente."""
    return np.maximum(d["ef"][n][fatia] / d["q"][n][fatia], d["sig"][n])


def _mede(modelo, C, S, td, banda, ibins_pico, t_pico_iia, T_lim=T_LIM):
    """Um passe completo das três medidas sobre uma realização de C(t)."""
    nomes = list(C)
    # SED do pico: média dos bins do pico, para não medir a cor num ponto só
    dC_p = {n: float(np.mean(C[n][ibins_pico]) - 1.0) for n in nomes}
    sg_p = {n: float(np.sqrt(np.sum(S[n][ibins_pico] ** 2)) / len(ibins_pico))
            for n in nomes}
    T_sed, A_sed, chi2_sed = modelo.temperatura_sed(dC_p, sg_p, T_lim)

    out = dict(T_pico_sed=T_sed, A_sed=A_sed, chi2_sed=chi2_sed,
               dC_pico=dC_p, sig_pico=sg_p)

    for tag, T_ref in (("IIa", t_pico_iia), ("IIb", T_sed)):
        if not np.isfinite(T_ref) or T_ref <= modelo.T_star or dC_p[banda] <= 0:
            out |= {f"A_{tag}": np.nan, f"E_{tag}": np.nan,
                    f"T_{tag}": np.full(td.size, modelo.T_star)}
            continue
        A_proj, A_dep = modelo.area_pico(dC_p[banda], T_ref, banda)
        T = modelo.temperatura(C[banda], A_proj, banda)
        E, L = modelo.energia(td, T, A_dep)
        out |= {f"A_{tag}": A_dep, f"E_{tag}": E, f"T_{tag}": T, f"L_{tag}": L}

    T_mb, A_mb, chi2_mb = modelo.ajuste_multibanda(C, S, T_lim)
    E_mb, L_mb = modelo.energia(td, T_mb, A_mb)
    out |= dict(T_MB=T_mb, A_MB=A_mb, chi2_MB=chi2_mb, E_MB=E_mb, L_MB=L_mb)
    return out


def analisa_flare(d, fl, mod=None, t_pico_iia=T_PICO_IIA, n_pico=N_PICO,
                  nmc=NMC, semente=SEMENTE):
    """Tudo de uma flare, do C(t) da janela às três energias e suas barras."""
    mod = mod or modelo
    banda = fl["banda_det"]
    fatia = slice(fl["j0"], fl["j1"])
    t = d["t"][fatia]
    td = t - t[0]
    nomes = d["bandas"]

    C = {n: d["C"][n][fatia] for n in nomes}
    S = {n: _sigma_C(d, n, fatia) for n in nomes}
    # nuvem e guiagem ruim ficam de fora: entram como quiescente, não como sinal
    bom = np.all([np.isfinite(C[n]) & ~d["ruim"][n][fatia] for n in nomes], axis=0)
    if bom.sum() < 3:
        raise ValueError(f"{fl['id']}: só {bom.sum()} bins com as quatro bandas")
    # buracos ficam no quiescente, para não abrir vão na integral da energia
    for n in nomes:
        C[n] = np.where(bom, C[n], 1.0)
        S[n] = np.where(bom, S[n], np.nanmax(S[n]))

    k_pico = int(np.argmax(np.where(bom, C[banda], -np.inf)))
    meio = n_pico // 2
    ibins = np.arange(max(k_pico - meio, 0),
                      min(k_pico - meio + n_pico, td.size))
    ibins = ibins[bom[ibins]]

    r = _mede(mod, C, S, td, banda, ibins, t_pico_iia)

    # ── incertezas: Monte Carlo sobre os erros das quatro curvas ────────────
    mc = {}
    if nmc:
        rng = np.random.default_rng(semente)
        chaves = ("T_pico_sed", "A_sed", "A_IIa", "E_IIa", "A_IIb", "E_IIb", "E_MB")
        amostras = {k: [] for k in chaves}
        for _ in range(int(nmc)):
            Ck = {n: C[n] + rng.normal(0.0, S[n]) for n in nomes}
            kk = int(np.argmax(np.where(bom, Ck[banda], -np.inf)))
            ib = np.arange(max(kk - meio, 0), min(kk - meio + n_pico, td.size))
            ib = ib[bom[ib]]
            try:
                rk = _mede(mod, Ck, S, td, banda, ib, t_pico_iia)
            except ValueError:
                continue
            for k in chaves:
                amostras[k].append(rk[k])
        for k, v in amostras.items():
            v = np.asarray(v, float)
            v = v[np.isfinite(v)]
            mc[k] = (np.percentile(v, [16, 50, 84]) if v.size > 10
                     else np.full(3, np.nan))

    gl = max(len(nomes) - 2, 1)
    return dict(fl, t=t, td=td, C=C, S=S, bom=bom, banda=banda, k_pico=k_pico,
                ibins_pico=ibins, mc=mc, t_pico_iia=t_pico_iia,
                chi2red_sed=r["chi2_sed"] / gl, **r)


def resumo_flare(r):
    """Uma flare em quinze linhas: o que foi medido, com o que foi assumido."""
    def barra(chave, valor, esc=1.0, fmt="{:.3g}"):
        v = r["mc"].get(chave)
        if v is None or not np.isfinite(v).all():
            return fmt.format(valor * esc)
        lo, med, hi = v * esc
        return f"{fmt.format(valor*esc)} (-{fmt.format(med-lo)} +{fmt.format(hi-med)})"

    ini = Time(r["t_ini"], format="jd").isot.replace("T", " ")[:19]
    print(f"── {r['id']} " + "─" * 56)
    print(f"início {ini} UTC | duração {r['dur_min']:.1f} min "
          f"({r['n_bins']} bins de {BIN_S:.0f} s) | pico {r['pico_sig']:.0f}σ em {r['banda']}")
    print("amplitude no pico:  " + " | ".join(
        f"{n} {r['amp'][n]*1e3:+7.2f} ppt ({r['amp_sig'][n]:4.1f}σ)" for n in r["C"]))
    print(f"duração equivalente (banda {r['banda']}): {r['ed_s']:.1f} s")
    print(f"base antes/depois: {r['n_pre']} bins a {r['base_pre_sig']:+.1f}σ | "
          f"{r['n_pos']} bins a {r['base_pos_sig']:+.1f}σ do quiescente")
    if r["n_pre"] < 8 or abs(np.nan_to_num(r["base_pre_sig"])) > 3:
        print("    ! base pré-flare curta ou deslocada: o quiescente sob a "
              "flare é interpolado a partir do que")
        print("      sobra, e a amplitude — logo a energia — herda essa "
              "escolha. Veja a célula 11.")
    print()
    print(f"T de pico MEDIDO (SED das {len(r['C'])} bandas): "
          f"{barra('T_pico_sed', r['T_pico_sed'], fmt='{:.0f}')} K"
          f"   [chi2 red. = {r['chi2red_sed']:.2f}]")
    print(f"área da SED no pico:      {barra('A_sed', r['A_sed'], fmt='{:.3e}')} cm²"
          f"  ({r['A_sed']/modelo.A_star*100:.3f}% do disco)")
    print()
    print(f"IIa (T assumido = {r['t_pico_iia']:.0f} K)")
    print(f"    área {barra('A_IIa', r['A_IIa'], fmt='{:.3e}')} cm² | "
          f"E = {barra('E_IIa', r['E_IIa'], fmt='{:.3e}')} erg")
    print(f"IIb (T de pico medido)         <- a medida")
    print(f"    área {barra('A_IIb', r['A_IIb'], fmt='{:.3e}')} cm² | "
          f"E = {barra('E_IIb', r['E_IIb'], fmt='{:.3e}')} erg")
    ajust = r["T_MB"] > modelo.T_star
    teto  = r["T_MB"] > 0.99 * T_LIM[1]
    print(f"MB  (T e área livres por instante)")
    print(f"    {ajust.sum()}/{r['T_MB'].size} bins passaram a porta de "
          f"{NSIG_MB:.0f}σ; {teto.sum()} deles no teto de {T_LIM[1]:.0f} K")
    print(f"    T máx {np.nanmax(r['T_MB']):.0f} K | "
          f"E = {barra('E_MB', r['E_MB'], fmt='{:.3e}')} erg")
    if np.isfinite(r["E_MB"]) and np.isfinite(r["E_IIb"]) and r["E_IIb"] > 0:
        print(f"    E_MB / E_IIb = {r['E_MB']/r['E_IIb']:.2f} — acima de ~1.5 o "
              f"excesso é ruído em T entrando como T⁴, não energia.")

In [14]:
# ═══ 8 · figuras ════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

DESLOC = 0.06     # deslocamento vertical entre bandas, na figura da noite


def plota_noite(d, flares=(), figsize=(14, 7)):
    """A noite inteira, banda a banda, com o quiescente e as flares marcadas."""
    t0 = d["t"][0]
    tm = (d["t"] - t0) * 1440
    fig, ax = plt.subplots(figsize=figsize)
    for k, n in enumerate(d["bandas"]):
        off = k * DESLOC
        ax.plot(tm, d["f"][n] + off, ".", ms=2, color=CORES[n], alpha=.55)
        ax.plot(tm, d["q"][n] + off, "-", lw=1.4, color="k", alpha=.75)
        ax.text(tm[-1] + 1.5, np.nanmedian(d["q"][n]) + off, n,
                color=CORES[n], fontsize=13, va="center", fontweight="bold")
    for fl in flares:
        ax.axvspan((d["t"][fl["i0"]] - t0) * 1440,
                   (d["t"][fl["i1"] - 1] - t0) * 1440,
                   color="0.55", alpha=.18, zorder=0)
        ax.text((d["t"][fl["ipico"]] - t0) * 1440,
                ax.get_ylim()[1], fl["id"].split("_")[-1],
                ha="center", va="top", fontsize=9, color="0.3")
    ax.set_xlabel(f"minutos desde {Time(t0, format='jd').isot[:19]} UTC")
    ax.set_ylabel("fluxo diferencial normalizado (+ deslocamento)")
    ax.set_title(f"AU Mic · {rotulo_noite(d['noite'])} · bins de {d['bin_s']:.0f} s")
    fig.tight_layout()
    return fig


def _modelo_banda(r, n, tag="IIb"):
    """dC previsto numa banda pela solução `tag` (T(t) + área única)."""
    A_proj = r[f"A_{tag}"] * modelo.mu
    return A_proj / modelo.A_star * (
        modelo.bandas[n].I(r[f"T_{tag}"]) / modelo.I_star[n] - 1.0)


def plota_flare(r, tag="IIb", figsize=(13, 9)):
    """Quatro painéis: curvas + modelo, T(t), área/luminosidade e a SED do pico."""
    tm = r["td"] * 1440
    fig, ax = plt.subplots(2, 2, figsize=figsize)
    a, b, c, e = ax[0, 0], ax[0, 1], ax[1, 0], ax[1, 1]

    for n in r["C"]:
        a.errorbar(tm, (r["C"][n] - 1) * 1e3, r["S"][n] * 1e3, fmt=".", ms=4,
                   lw=.6, color=CORES[n], alpha=.7, label=n)
        a.plot(tm, _modelo_banda(r, n, tag) * 1e3, "-", lw=1.6, color=CORES[n])
    a.axhline(0, color="k", lw=.8)
    a.axvspan(tm[r["ibins_pico"][0]], tm[r["ibins_pico"][-1]],
              color="0.6", alpha=.15)
    a.set_ylabel("C - 1  [ppt]"); a.set_xlabel("minutos desde o início da janela")
    a.legend(ncol=4, fontsize=10); a.set_title(f"{r['id']} · pontos e modelo {tag}")

    b.plot(tm, r["T_MB"], ".-", ms=4, lw=.8, color="0.55", label="MB (por instante)")
    b.plot(tm, r["T_IIb"], "-", lw=1.8, color="crimson", label="IIb")
    b.plot(tm, r["T_IIa"], "--", lw=1.4, color="tab:blue",
           label=f"IIa (T assumido {r['t_pico_iia']:.0f} K)")
    b.axhline(r["T_pico_sed"], color="crimson", ls=":", lw=1.2,
              label=f"T pico SED = {r['T_pico_sed']:.0f} K")
    b.axhline(modelo.T_star, color="k", lw=.8)
    # o eixo segue a solução IIb: sem isso os poucos instantes que o MB manda
    # para o teto de T_LIM esmagam a parte legível do painel
    topo = min(T_LIM[1], 2.2 * max(r["T_pico_sed"], np.nanmax(r["T_IIb"])))
    fora = int((r["T_MB"] > topo).sum())
    b.set_ylim(modelo.T_star * 0.9, topo)
    b.set_ylabel("T da flare [K]"); b.set_xlabel("minutos"); b.legend(fontsize=9)
    b.set_title("temperatura" + (f" · {fora} pontos MB acima do eixo" if fora else ""))

    c.plot(tm, r["A_MB"], ".-", ms=4, lw=.8, color="0.55", label="área MB")
    c.axhline(r[f"A_{tag}"], color="crimson", lw=1.6, label=f"área {tag} (fixa)")
    c.set_yscale("log"); c.set_ylabel("área deprojetada [cm²]")
    c.set_xlabel("minutos"); c.legend(fontsize=9)
    cc = c.twinx()
    cc.plot(tm, r.get(f"L_{tag}", np.zeros_like(tm)), "-", lw=1.2, color="tab:orange")
    cc.set_ylabel("L da flare [erg/s]", color="tab:orange")
    c.set_title("área e luminosidade")

    lam = np.array([modelo.bandas[n].lambda_efetivo() for n in r["C"]]) * 1e9
    dC = np.array([r["dC_pico"][n] for n in r["C"]]) * 1e3
    sg = np.array([r["sig_pico"][n] for n in r["C"]]) * 1e3
    e.errorbar(lam, dC, sg, fmt="o", color="k", ms=7, label="medido no pico")
    for T, cor, est in ((r["T_pico_sed"], "crimson", "-"),
                        (r["t_pico_iia"], "tab:blue", "--")):
        A = modelo.area_pico(r["dC_pico"][r["banda"]], T, r["banda"])[0]
        prev = [A / modelo.A_star * (modelo.bandas[n].I(T) / modelo.I_star[n] - 1)
                for n in r["C"]]
        e.plot(lam, np.array(prev) * 1e3, est, color=cor, lw=1.6,
               label=f"corpo negro {T:.0f} K")
    e.set_xlabel("λ efetivo da banda [nm]"); e.set_ylabel("C - 1 no pico [ppt]")
    e.legend(fontsize=9); e.set_title("SED do pico")

    fig.tight_layout()
    return fig

In [21]:
# ═══ 9 · a noite, de ponta a ponta ══════════════════════════════════════════
# Curva diferencial -> quiescente -> detecção. Nada aqui é ajustado: as escolhas
# estão todas na célula 1.
noites, deteccoes = {}, {}
for noite in NOITES_USO:
    d = prepara_noite(noite)
    resumo_noite(d)
    fl = detecta_flares(d)
    print(f"  -> {len(fl)} flare(s) acima de {NSIG_PICO:.0f}σ em pelo menos "
          f"{NBANDAS_MIN} bandas")
    if fl:
        display(tabela_deteccao(fl).style.format({
            "dur (min)": "{:.1f}", "pico (σ)": "{:.0f}", "ED (s)": "{:.1f}",
            **{f"amp {n} (ppt)": "{:+.1f}" for n in d["bandas"]}}).hide(axis="index"))
    noites[noite], deteccoes[noite] = d, fl
    plota_noite(d, fl)
    plt.show()
    print()

── 2026-08-15 ────────────────────────────────────────────────────
2.70 h | grade de 15 s -> 649 bins (631 com as 4 bandas boas) | quiescente: mediana
  g: cadência  4.12 s | σ quiescente  2.73 ppt | C máx 1.1819 | comparações [1, 3, 2]
  r: cadência  1.32 s | σ quiescente  3.24 ppt | C máx 1.0617 | comparações [1, 2, 3]
  i: cadência  1.32 s | σ quiescente  3.67 ppt | C máx 1.0236 | comparações [1, 2, 3]
  z: cadência  1.32 s | σ quiescente  4.31 ppt | C máx 1.0186 | comparações [1, 2, 3]
  -> 1 flare(s) acima de 5σ em pelo menos 2 bandas


flare,início (UTC),dur (min),bins,pico (σ),ED (s),amp g (ppt),amp r (ppt),amp i (ppt),amp z (ppt)
20260815_F01,22:38:10,15.0,61,68,49.0,+181.9,+50.8,+8.4,-4.1


In [22]:
# ═══ 10 · temperatura e energia de cada flare ═══════════════════════════════
# `analisa_flare` roda os três métodos e o Monte Carlo das barras de erro.
resultados = {}
for noite in NOITES_USO:
    for fl in deteccoes[noite]:
        r = analisa_flare(noites[noite], fl)
        resumo_flare(r)
        resultados[fl["id"]] = r
        plota_flare(r)
        plt.show()
        print()


def tabela_flares(res=None):
    """As flares lado a lado — é esta tabela que vai para o texto."""
    res = res if res is not None else resultados
    linhas = []
    for r in res.values():
        def mc(chave, valor):
            v = r["mc"].get(chave)
            return (np.nan if v is None or not np.isfinite(v).all()
                    else 0.5 * (v[2] - v[0]))
        linhas.append({
            "flare": r["id"],
            "início (UTC)": Time(r["t_ini"], format="jd").isot[11:19],
            "dur (min)": r["dur_min"],
            "amp g (ppt)": r["amp"][r["banda"]] * 1e3,
            "ED (s)": r["ed_s"],
            "T pico (K)": r["T_pico_sed"], "σT (K)": mc("T_pico_sed", r["T_pico_sed"]),
            "chi2 red": r["chi2red_sed"],
            "área (cm²)": r["A_IIb"], "σA (cm²)": mc("A_IIb", r["A_IIb"]),
            "E IIb (erg)": r["E_IIb"], "σE (erg)": mc("E_IIb", r["E_IIb"]),
            "E IIa (erg)": r["E_IIa"], "E MB (erg)": r["E_MB"]})
    return pd.DataFrame(linhas)


TAB = tabela_flares()
FMT = {"dur (min)": "{:.1f}", "amp g (ppt)": "{:+.1f}", "ED (s)": "{:.1f}",
       "T pico (K)": "{:.0f}", "σT (K)": "{:.0f}", "chi2 red": "{:.2f}",
       "área (cm²)": "{:.2e}", "σA (cm²)": "{:.1e}", "E IIb (erg)": "{:.2e}",
       "σE (erg)": "{:.1e}", "E IIa (erg)": "{:.2e}", "E MB (erg)": "{:.2e}"}
print("── flares medidas " + "─" * 55)
display(TAB.style.format(FMT).hide(axis="index"))

── 20260815_F01 ────────────────────────────────────────────────────────
início 2026-08-15 22:38:10 UTC | duração 15.0 min (61 bins de 15 s) | pico 68σ em g
amplitude no pico:  g +181.86 ppt (68.1σ) | r  +50.78 ppt (16.0σ) | i   +8.43 ppt ( 2.3σ) | z   -4.09 ppt (-1.0σ)
duração equivalente (banda g): 49.0 s
base antes/depois: 8 bins a -6.4σ | 8 bins a +1.2σ do quiescente
    ! base pré-flare curta ou deslocada: o quiescente sob a flare é interpolado a partir do que
      sobra, e a amplitude — logo a energia — herda essa escolha. Veja a célula 11.

T de pico MEDIDO (SED das 4 bandas): 11083 (-893 +980) K   [chi2 red. = 11.79]
área da SED no pico:      4.892e+18 (-1.140e+18 +1.384e+18) cm²  (0.057% do disco)

IIa (T assumido = 10000 K)
    área 6.816e+18 (-2.423e+17 +1.074e+17) cm² | E = 1.028e+33 (-1.489e+31 +1.311e+31) erg
IIb (T de pico medido)         <- a medida
    área 4.871e+18 (-1.137e+18 +1.366e+18) cm² | E = 1.053e+33 (-2.598e+31 +4.622e+31) erg
MB  (T e área livres por insta

flare,início (UTC),dur (min),amp g (ppt),ED (s),T pico (K),σT (K),chi2 red,área (cm²),σA (cm²),E IIb (erg),σE (erg),E IIa (erg),E MB (erg)
20260815_F01,22:38:10,15.0,+181.9,49.0,11083,936,11.79,4.87e+18,1.3e+18,1.05e+33,3.6e+31,1.03e+33,2.92e+33


In [23]:
# ═══ 11 · robustez: o quanto o resultado depende das escolhas ═══════════════
# A barra Monte Carlo da célula 10 é só o ruído fotométrico. O que costuma
# dominar é a SISTEMÁTICA das escolhas: quiescente, abertura, grade e T assumido.
# Aqui cada uma é trocada de uma vez, e o efeito em T de pico e energia aparece
# em coluna própria.

def robustez(noite, variantes=None):
    variantes = variantes or [
        dict(rotulo="referência"),
        dict(rotulo="quiescente poly", modo_q="poly"),
        dict(rotulo=f"janela quiescente {JANELA_Q/2:.0f} min",
             janela_q=JANELA_Q / 2),
        dict(rotulo=f"janela quiescente {JANELA_Q*2:.0f} min",
             janela_q=JANELA_Q * 2),
        dict(rotulo="abertura 5 px", aper=5),
        dict(rotulo="abertura 12 px", aper=12),
        dict(rotulo="2 comparações", ncomps=2),
        dict(rotulo=f"grade {BIN_S*2:.0f} s", bin_s=BIN_S * 2),
        dict(rotulo="T_IIa = 15 000 K", t_pico_iia=15000.0),
    ]
    linhas = []
    for v in variantes:
        rot = v["rotulo"]
        prep = {k: v[k] for k in ("bin_s", "aper", "ncomps", "modo_q", "janela_q")
                if k in v}
        try:
            d = prepara_noite(noite, **prep)
            fls = detecta_flares(d)
            if not fls:
                linhas.append({"variante": rot, "flares": 0})
                continue
            # a de maior amplitude, para comparar sempre a mesma
            fl = max(fls, key=lambda x: x["amp"][x["banda_det"]])
            r = analisa_flare(d, fl, t_pico_iia=v.get("t_pico_iia", T_PICO_IIA),
                              nmc=0)
            linhas.append({
                "variante": rot, "flares": len(fls), "dur (min)": r["dur_min"],
                "amp g (ppt)": r["amp"][r["banda"]] * 1e3,
                "T pico (K)": r["T_pico_sed"], "chi2 red": r["chi2red_sed"],
                "área (cm²)": r["A_IIb"], "E IIb (erg)": r["E_IIb"],
                "E IIa (erg)": r["E_IIa"]})
        except Exception as exc:
            linhas.append({"variante": rot, "flares": np.nan,
                           "erro": f"{type(exc).__name__}: {exc}"})
    tab = pd.DataFrame(linhas)
    if "E IIb (erg)" in tab and np.isfinite(tab["E IIb (erg)"]).any():
        ref = tab["E IIb (erg)"].iloc[0]
        tab["E / E_ref"] = tab["E IIb (erg)"] / ref
    return tab


ROBUSTEZ = {}
for noite in NOITES_USO:
    if not deteccoes[noite]:
        continue
    tab = robustez(noite)
    ROBUSTEZ[noite] = tab
    print(f"── robustez · {rotulo_noite(noite)} " + "─" * 40)
    display(tab.style.format({
        "dur (min)": "{:.1f}", "amp g (ppt)": "{:+.1f}", "T pico (K)": "{:.0f}",
        "chi2 red": "{:.2f}", "área (cm²)": "{:.2e}", "E IIb (erg)": "{:.2e}",
        "E IIa (erg)": "{:.2e}", "E / E_ref": "{:.2f}"},
        na_rep="—").hide(axis="index"))
    e = tab["E IIb (erg)"].dropna()
    if len(e) > 1:
        print(f"espalhamento da energia entre as variantes: "
              f"{(e.max()/e.min()-1)*100:.0f}%  — compare com a barra Monte Carlo "
              f"da célula 10 antes de citar uma incerteza.")
    print()

── robustez · 2026-08-15 ────────────────────────────────────────


variante,flares,dur (min),amp g (ppt),T pico (K),chi2 red,área (cm²),E IIb (erg),E IIa (erg),E / E_ref
referência,1,15.0,+181.9,11083,11.79,4.87e+18,1.05e+33,1.03e+33,1.00
quiescente poly,1,19.0,+185.1,9490,10.06,8.33e+18,1.19e+33,1.19e+33,1.13
janela quiescente 10 min,1,8.7,+157.1,8008,11.03,1.35e+19,6.50e+32,6.43e+32,0.62
janela quiescente 40 min,1,22.2,+198.0,10138,13.82,7.10e+18,1.43e+33,1.43e+33,1.36
abertura 5 px,1,13.5,+167.3,6803,1.09,2.93e+19,9.86e+32,8.82e+32,0.94
abertura 12 px,1,15.0,+185.0,13770,6.65,2.65e+18,1.22e+33,1.05e+33,1.15
2 comparações,1,14.7,+180.2,10418,10.25,5.83e+18,9.76e+32,9.69e+32,0.93
grade 30 s,1,18.5,+179.2,10173,14.52,6.23e+18,1.11e+33,1.11e+33,1.05
T_IIa = 15 000 K,1,15.0,+181.9,11083,11.79,4.87e+18,1.05e+33,1.28e+33,1.00


espalhamento da energia entre as variantes: 121%  — compare com a barra Monte Carlo da célula 10 antes de citar uma incerteza.



In [24]:
# ═══ 12 · arquivos gravados ═════════════════════════════════════════════════
# Curvas de cada flare nas quatro bandas com T(t) e L(t), a tabela das flares,
# a tabela de robustez e um resumo em texto.

def salva_flare(r, prefixo):
    """C(t), erro, modelo e as soluções de T e área, banda a banda."""
    cols = {"t_bjd_tdb": r["t"], "minutos": r["td"] * 1440}
    for n in r["C"]:
        cols[f"C_{n}"] = r["C"][n]
        cols[f"sigC_{n}"] = r["S"][n]
        cols[f"modelo_IIb_{n}"] = 1.0 + _modelo_banda(r, n, "IIb")
    cols |= {"T_IIa_K": r["T_IIa"], "T_IIb_K": r["T_IIb"], "T_MB_K": r["T_MB"],
             "A_MB_cm2": r["A_MB"], "L_IIb_erg_s": r["L_IIb"],
             "L_MB_erg_s": r["L_MB"], "bins_bons": r["bom"].astype(int)}
    caminho = f"{prefixo}_{r['id']}_curvas.csv"
    pd.DataFrame(cols).to_csv(caminho, index=False, float_format="%.8g")
    return caminho


def texto(tab, fmt=None):
    return tab.to_string(index=False, float_format=lambda v: f"{v:.4g}",
                         na_rep="—")


PREFIXO = os.path.join(OUTDIR, "aumic_flares")
gravados = []

for r in resultados.values():
    gravados.append(salva_flare(r, PREFIXO))

TAB.to_csv(f"{PREFIXO}_tabela.csv", index=False, float_format="%.6g")
gravados.append(f"{PREFIXO}_tabela.csv")

for noite, tab in ROBUSTEZ.items():
    caminho = f"{PREFIXO}_{noite}_robustez.csv"
    tab.to_csv(caminho, index=False, float_format="%.6g")
    gravados.append(caminho)

resumo_txt = f"{PREFIXO}_resumo.txt"
with open(resumo_txt, "w", encoding="utf-8") as fh:
    fh.write("Flares de AU Mic no SPARC4 — Heinzel+2026\n")
    fh.write(f"T* = {T_STAR:.0f} K | R* = {R_STAR} R_sol | mu = {MU} | "
             f"A* = {modelo.A_star:.4e} cm2\n")
    fh.write(f"resposta: {os.path.basename(RESPOSTA)} | abertura {APER} px | "
             f"{NCOMPS} comparações | feixe {BEAM}\n")
    fh.write(f"grade {BIN_S:.0f} s | quiescente {QUIESC} ({JANELA_Q:.0f} min) | "
             f"detecção em {BANDA_DET} a {NSIG_PICO:.0f} sigma\n")
    fh.write(f"noites: {', '.join(rotulo_noite(n) for n in NOITES_USO)}\n\n")
    fh.write("flares medidas\n" + texto(TAB) + "\n\n")
    for noite, tab in ROBUSTEZ.items():
        fh.write(f"robustez · {rotulo_noite(noite)}\n" + texto(tab) + "\n\n")
    fh.write("E IIb é a medida (T de pico medido na SED). E IIa depende do T\n"
             "assumido e E MB é limite superior: o ruído em T entra como T^4.\n")
gravados.append(resumo_txt)

print(f"{len(gravados)} arquivos em {OUTDIR}:")
for g in gravados:
    print("  " + os.path.basename(g))

4 arquivos em C:/Users/paola/Desktop/Doutorado/Codigos-doc/Flares-/novo_homepage/Sparc4-data\resultados_flares:
  aumic_flares_20260815_F01_curvas.csv
  aumic_flares_tabela.csv
  aumic_flares_20260815_robustez.csv
  aumic_flares_resumo.txt


In [25]:

%matplotlib qt

# ═══ 13 · temperatura e energia banda a banda ═══════════════════════════════
# Até aqui a medida usa UMA banda para a área (a de detecção) e as QUATRO para o
# T de pico. Esta célula refaz a conta com cada banda isolada, de dois jeitos:
#
#   área própria  a área sai da amplitude DAQUELA banda (Eq. 6) e T(t) da mesma
#                 banda (Eq. 7) — é exatamente o que se teria com fotometria de
#                 banda única, e a dispersão entre as quatro é a sistemática
#                 que o SPARC4 permite medir e que TESS/Kepler não veem.
#   área comum    a área é a da SED no pico, a MESMA para as quatro; cada banda
#                 só inverte a Eq. (7). Aí as quatro T(t) medem a mesma coisa e
#                 a discordância entre elas testa a hipótese de corpo negro.
#
# O T de pico do IIb continua vindo da SED das quatro bandas: uma banda sozinha
# não mede T, ela só converte uma amplitude num par (área, T) já escolhido.

NMC_BANDA = 200          # realizações do Monte Carlo desta célula (0 desliga)


def _passe_bandas(mod, C, S, td, ibins, t_iia, T_lim=T_LIM):
    """Um passe das duas variantes, em todas as bandas, sobre uma realização."""
    nomes = list(C)
    dC_p = {n: float(np.mean(C[n][ibins]) - 1.0) for n in nomes}
    sg_p = {n: float(np.sqrt(np.sum(S[n][ibins] ** 2)) / len(ibins)) for n in nomes}
    T_sed, A_sed, chi2 = mod.temperatura_sed(dC_p, sg_p, T_lim)

    out = dict(T_pico_sed=T_sed, A_sed=A_sed, chi2_sed=chi2)
    for n in nomes:
        o = dict(dC_pico=dC_p[n], sig_pico=sg_p[n],
                 snr=dC_p[n] / sg_p[n] if sg_p[n] > 0 else np.nan)
        # ── área própria: IIa (T assumido) e IIb (T de pico medido na SED) ──
        for tag, T_ref in (("IIa", t_iia), ("IIb", T_sed)):
            if not np.isfinite(T_ref) or T_ref <= mod.T_star or dC_p[n] <= 0:
                o |= {f"A_{tag}": np.nan, f"E_{tag}": np.nan,
                      f"T_{tag}": np.full(td.size, mod.T_star)}
                continue
            A_proj, A_dep = mod.area_pico(dC_p[n], T_ref, n)
            T = mod.temperatura(C[n], A_proj, n)
            E, L = mod.energia(td, T, A_dep)
            o |= {f"A_{tag}": A_dep, f"E_{tag}": E, f"T_{tag}": T, f"L_{tag}": L}
        # ── área comum: a da SED no pico (deprojetada -> projetada = *mu) ───
        if np.isfinite(A_sed) and A_sed > 0:
            T = mod.temperatura(C[n], A_sed * mod.mu, n)
            E, L = mod.energia(td, T, A_sed)
            o |= dict(A_com=A_sed, T_com=T, E_com=E, L_com=L)
        else:
            o |= dict(A_com=np.nan, T_com=np.full(td.size, mod.T_star), E_com=np.nan)
        out[n] = o
    return out


def por_banda(r, mod=None, t_pico_iia=None, nmc=NMC_BANDA, semente=SEMENTE):
    """Temperatura, área e energia de uma flare, banda a banda.

    Devolve o mesmo dicionário de `_passe_bandas` acrescido de `mc[banda][chave]`
    com os percentis 16/50/84 de cada escalar, do mesmo Monte Carlo da célula 7:
    as quatro curvas são reperturbadas juntas, então o T da SED varia junto e a
    barra do IIb já inclui a incerteza do T de pico.
    """
    mod = mod or modelo
    t_iia = r["t_pico_iia"] if t_pico_iia is None else t_pico_iia
    C, S, td, ib = r["C"], r["S"], r["td"], r["ibins_pico"]

    out = _passe_bandas(mod, C, S, td, ib, t_iia)

    chaves = ("A_IIa", "E_IIa", "A_IIb", "E_IIb", "E_com")
    mc = {n: {k: np.full(3, np.nan) for k in chaves} for n in C}
    mc["T_pico_sed"] = np.full(3, np.nan)
    if nmc:
        rng = np.random.default_rng(semente)
        acum = {n: {k: [] for k in chaves} for n in C}
        acum_T = []
        meio = len(ib) // 2
        for _ in range(int(nmc)):
            Ck = {n: C[n] + rng.normal(0.0, S[n]) for n in C}
            kk = int(np.argmax(np.where(r["bom"], Ck[r["banda"]], -np.inf)))
            ibk = np.arange(max(kk - meio, 0), min(kk - meio + len(ib), td.size))
            ibk = ibk[r["bom"][ibk]]
            if ibk.size == 0:
                continue
            try:
                ok = _passe_bandas(mod, Ck, S, td, ibk, t_iia)
            except ValueError:
                continue
            acum_T.append(ok["T_pico_sed"])
            for n in C:
                for k in chaves:
                    acum[n][k].append(ok[n][k])
        def pct(v):
            v = np.asarray(v, float); v = v[np.isfinite(v)]
            return np.percentile(v, [16, 50, 84]) if v.size > 10 else np.full(3, np.nan)
        mc["T_pico_sed"] = pct(acum_T)
        for n in C:
            for k in chaves:
                mc[n][k] = pct(acum[n][k])
    out["mc"] = mc
    return out


def tabela_por_banda(r, pb=None):
    """Uma linha por banda: amplitude, área, T máximo e energia nas variantes."""
    pb = pb or por_banda(r)
    linhas = []
    for n in r["C"]:
        o, m = pb[n], pb["mc"][n]
        linhas.append({
            "banda": n,
            "λ_ef (nm)": modelo.bandas[n].lambda_efetivo() * 1e9,
            "amp (ppt)": o["dC_pico"] * 1e3, "S/R": o["snr"],
            "A IIa (cm²)": o["A_IIa"], "A IIb (cm²)": o["A_IIb"],
            "T máx IIb (K)": np.nanmax(o["T_IIb"]),
            "T máx área comum (K)": np.nanmax(o["T_com"]),
            "E IIa (erg)": o["E_IIa"], "σE IIa": 0.5 * (m["E_IIa"][2] - m["E_IIa"][0]),
            "E IIb (erg)": o["E_IIb"], "σE IIb": 0.5 * (m["E_IIb"][2] - m["E_IIb"][0]),
            "E área comum (erg)": o["E_com"]})
    tab = pd.DataFrame(linhas)
    ref = tab.loc[tab["banda"] == r["banda"], "E IIb (erg)"]
    if len(ref) and np.isfinite(ref.iloc[0]) and ref.iloc[0] > 0:
        tab["E / E(" + r["banda"] + ")"] = tab["E IIb (erg)"] / ref.iloc[0]
    return tab


def plota_por_banda(r, pb=None, figsize=(13, 9)):
    """T(t) nas duas variantes + área e energia banda a banda."""
    pb = pb or por_banda(r)
    tm = r["td"] * 1440
    nomes = list(r["C"])
    x = np.arange(len(nomes))
    fig, ax = plt.subplots(2, 2, figsize=figsize)
    a, b, c, e = ax[0, 0], ax[0, 1], ax[1, 0], ax[1, 1]

    # (a) T(t) com a área PRÓPRIA de cada banda
    for n in nomes:
        a.plot(tm, pb[n]["T_IIb"], "-", lw=1.6, color=CORES[n], label=n)
    a.axhline(pb["T_pico_sed"], color="k", ls=":", lw=1.2,
              label=f"T pico SED = {pb['T_pico_sed']:.0f} K")
    a.axhline(modelo.T_star, color="k", lw=.8)
    a.set_ylabel("T da flare [K]"); a.set_xlabel("minutos")
    a.legend(ncol=3, fontsize=9)
    a.set_title("T(t) · área própria de cada banda (IIb)")

    # (b) T(t) com a MESMA área nas quatro: teste do corpo negro
    for n in nomes:
        b.plot(tm, pb[n]["T_com"], "-", lw=1.6, color=CORES[n], label=n)
    b.plot(tm, r["T_MB"], ".", ms=3, color="0.6", label="MB (4 bandas juntas)")
    b.axhline(modelo.T_star, color="k", lw=.8)
    topo = min(T_LIM[1], 1.25 * max([np.nanmax(pb[n]["T_com"]) for n in nomes]))
    b.set_ylim(modelo.T_star * .9, topo)
    b.set_ylabel("T da flare [K]"); b.set_xlabel("minutos")
    b.legend(ncol=3, fontsize=9)
    b.set_title(f"T(t) · área comum = {pb['A_sed']:.2e} cm² (teste do corpo negro)")

    # (c) área por banda
    for tag, mk, cor in (("IIa", "o", "tab:blue"), ("IIb", "s", "crimson")):
        y = np.array([pb[n][f"A_{tag}"] for n in nomes])
        er = np.array([[max(pb[n][f"A_{tag}"] - pb["mc"][n][f"A_{tag}"][0], 0),
                        max(pb["mc"][n][f"A_{tag}"][2] - pb[n][f"A_{tag}"], 0)]
                       for n in nomes]).T
        er = np.where(np.isfinite(er), er, 0.0)
        c.errorbar(x + (0.08 if tag == "IIb" else -0.08), y, er, fmt=mk, ms=8,
                   color=cor, capsize=3,
                   label=f"{tag}" + (f" (T={r['t_pico_iia']:.0f} K)" if tag == "IIa" else ""))
    c.axhline(pb["A_sed"], color="k", ls="--", lw=1.2, label="área da SED")
    c.set_xticks(x); c.set_xticklabels(nomes); c.set_yscale("log")
    c.set_ylabel("área deprojetada [cm²]"); c.legend(fontsize=9)
    c.set_title("área da Eq. (6), banda a banda")

    # (d) energia por banda
    for tag, mk, cor in (("IIa", "o", "tab:blue"), ("IIb", "s", "crimson"),
                         ("com", "^", "0.45")):
        y = np.array([pb[n][f"E_{tag}"] for n in nomes])
        if tag == "com":
            e.plot(x, y, mk, ms=8, mfc="none", color=cor, label="área comum")
            continue
        er = np.array([[max(pb[n][f"E_{tag}"] - pb["mc"][n][f"E_{tag}"][0], 0),
                        max(pb["mc"][n][f"E_{tag}"][2] - pb[n][f"E_{tag}"], 0)]
                       for n in nomes]).T
        er = np.where(np.isfinite(er), er, 0.0)
        e.errorbar(x + (0.08 if tag == "IIb" else -0.08), y, er, fmt=mk, ms=8,
                   color=cor, capsize=3, label=tag)
    if np.isfinite(r.get("E_IIb", np.nan)):
        e.axhline(r["E_IIb"], color="k", ls="--", lw=1.2,
                  label=f"medida da célula 10 ({r['banda']})")
    e.set_xticks(x); e.set_xticklabels(nomes); e.set_yscale("log")
    e.set_ylabel("energia bolométrica [erg]"); e.legend(fontsize=9)
    e.set_title("energia, banda a banda")

    fig.suptitle(f"{r['id']} · cada banda como se fosse a única", y=1.0)
    fig.tight_layout()
    return fig


POR_BANDA, TAB_BANDA = {}, {}
FMT_B = {"λ_ef (nm)": "{:.0f}", "amp (ppt)": "{:+.2f}", "S/R": "{:.1f}",
         "A IIa (cm²)": "{:.2e}", "A IIb (cm²)": "{:.2e}",
         "T máx IIb (K)": "{:.0f}", "T máx área comum (K)": "{:.0f}",
         "E IIa (erg)": "{:.2e}", "σE IIa": "{:.1e}",
         "E IIb (erg)": "{:.2e}", "σE IIb": "{:.1e}",
         "E área comum (erg)": "{:.2e}"}

for fid, r in resultados.items():
    pb = por_banda(r)
    tab = tabela_por_banda(r, pb)
    POR_BANDA[fid], TAB_BANDA[fid] = pb, tab
    print(f"── {fid} · banda a banda " + "─" * 40)
    fmt = dict(FMT_B)
    fmt |= {c: "{:.2f}" for c in tab.columns if c.startswith("E / E(")}
    display(tab.style.format(fmt, na_rep="—").hide(axis="index"))
    Ei = tab["E IIb (erg)"].to_numpy(float)
    Ei = Ei[np.isfinite(Ei) & (Ei > 0)]
    if Ei.size > 1:
        print(f"   espalhamento da energia entre as bandas: "
              f"{(Ei.max()/Ei.min()-1)*100:.0f}%  — é a sistemática de banda "
              f"única, e a barra Monte Carlo NÃO a contém.")
    tab.to_csv(f"{PREFIXO}_{fid}_bandas.csv", index=False, float_format="%.6g")
    plota_por_banda(r, pb)
    plt.show()
    print()

── 20260815_F01 · banda a banda ────────────────────────────────────────


banda,λ_ef (nm),amp (ppt),S/R,A IIa (cm²),A IIb (cm²),T máx IIb (K),T máx área comum (K),E IIa (erg),σE IIa,E IIb (erg),σE IIb,E área comum (erg),E / E(g)
g,457,+177.31,62.3,6.82e+18,4.87e+18,11174,11158,1.03e+33,1.4e+31,1.05e+33,3.2e+31,1.05e+33,1.00
r,614,+55.58,17.8,7.82e+18,6.00e+18,11569,12634,1.15e+33,4.0e+31,1.28e+33,1.4e+32,1.42e+33,1.22
i,753,+13.19,4.7,3.76e+18,2.99e+18,14863,11549,8.13e+32,2.6e+32,9.80e+32,5.3e+32,6.83e+32,0.93
z,894,+7.36,2.5,3.40e+18,2.75e+18,18941,13329,1.53e+33,2.1e+33,1.99e+33,3.5e+33,1.04e+33,1.89


   espalhamento da energia entre as bandas: 103%  — é a sistemática de banda única, e a barra Monte Carlo NÃO a contém.



In [27]:
# ═══ 13 · temperatura e energia banda a banda ═══════════════════════════════
# Até aqui a medida usa UMA banda para a área (a de detecção) e as QUATRO para o
# T de pico. Esta célula refaz a conta com cada banda isolada, de dois jeitos:
#
#   área própria  a área sai da amplitude DAQUELA banda (Eq. 6) e T(t) da mesma
#                 banda (Eq. 7) — é o que se teria com fotometria de banda
#                 única, e a dispersão entre as quatro é a sistemática que o
#                 SPARC4 mede e que TESS/Kepler não veem.
#   área comum    a área é a da SED no pico, a MESMA para as quatro; cada banda
#                 só inverte a Eq. (7). Aí as quatro T(t) medem a mesma coisa e
#                 a discordância entre elas testa a hipótese de corpo negro.
#
# O T de pico do IIb continua vindo da SED das quatro bandas: uma banda sozinha
# não mede T — ela só converte uma amplitude num par (área, T) já escolhido.

NMC_BANDA = 200      # realizações do Monte Carlo desta célula (0 desliga)
ORDEM_BANDAS = None  # None = ordem de λ; ou fixe, p.ex. ["g", "i", "r", "z"]


def _ordem(r):
    nomes = list(r["C"])
    if ORDEM_BANDAS:
        return [n for n in ORDEM_BANDAS if n in nomes]
    return sorted(nomes, key=lambda n: modelo.bandas[n].lambda_efetivo())


def _cum(t_dias, L):
    """Energia acumulada: ∫L dt do início da janela até cada instante."""
    t = np.asarray(t_dias, float) * 86400.0
    L = np.asarray(L, float)
    return np.concatenate([[0.0], np.cumsum(0.5 * (L[1:] + L[:-1]) * np.diff(t))])


def _prev(mod, n, A_dep, T):
    """dC previsto na banda n por um par (área deprojetada, T(t))."""
    return A_dep * mod.mu / mod.A_star * (mod.bandas[n].I(T) / mod.I_star[n] - 1.0)


def _passe_bandas(mod, C, S, td, ibins, t_iia, T_lim=T_LIM):
    """Um passe das duas variantes, em todas as bandas, sobre uma realização."""
    nomes = list(C)
    dC_p = {n: float(np.mean(C[n][ibins]) - 1.0) for n in nomes}
    sg_p = {n: float(np.sqrt(np.sum(S[n][ibins] ** 2)) / len(ibins)) for n in nomes}
    T_sed, A_sed, chi2 = mod.temperatura_sed(dC_p, sg_p, T_lim)

    out = dict(T_pico_sed=T_sed, A_sed=A_sed, chi2_sed=chi2)
    for n in nomes:
        o = dict(dC_pico=dC_p[n], sig_pico=sg_p[n],
                 snr=dC_p[n] / sg_p[n] if sg_p[n] > 0 else np.nan)
        # ── área própria: IIa (T assumido) e IIb (T de pico medido na SED) ──
        for tag, T_ref in (("IIa", t_iia), ("IIb", T_sed)):
            if not np.isfinite(T_ref) or T_ref <= mod.T_star or dC_p[n] <= 0:
                o |= {f"A_{tag}": np.nan, f"E_{tag}": np.nan,
                      f"T_{tag}": np.full(td.size, mod.T_star),
                      f"L_{tag}": np.zeros(td.size)}
                continue
            A_proj, A_dep = mod.area_pico(dC_p[n], T_ref, n)
            T = mod.temperatura(C[n], A_proj, n)
            E, L = mod.energia(td, T, A_dep)
            o |= {f"A_{tag}": A_dep, f"E_{tag}": E, f"T_{tag}": T, f"L_{tag}": L}
        # ── área comum: a da SED no pico (deprojetada → projetada = ×mu) ────
        if np.isfinite(A_sed) and A_sed > 0:
            T = mod.temperatura(C[n], A_sed * mod.mu, n)
            E, L = mod.energia(td, T, A_sed)
            o |= dict(A_com=A_sed, T_com=T, E_com=E, L_com=L)
        else:
            o |= dict(A_com=np.nan, T_com=np.full(td.size, mod.T_star),
                      E_com=np.nan, L_com=np.zeros(td.size))
        out[n] = o
    return out


def por_banda(r, mod=None, t_pico_iia=None, nmc=NMC_BANDA, semente=SEMENTE):
    """Temperatura, área e energia de uma flare, banda a banda.

    Acrescenta `mc[banda][chave]` (percentis 16/50/84) e `amostras[banda][chave]`
    (as realizações inteiras, para os histogramas). O Monte Carlo é o mesmo da
    célula 7: as quatro curvas são reperturbadas JUNTAS, então o T da SED varia
    junto e a barra do IIb já inclui a incerteza do T de pico.
    """
    mod = mod or modelo
    t_iia = r["t_pico_iia"] if t_pico_iia is None else t_pico_iia
    C, S, td, ib = r["C"], r["S"], r["td"], r["ibins_pico"]

    out = _passe_bandas(mod, C, S, td, ib, t_iia)

    chaves = ("A_IIa", "E_IIa", "A_IIb", "E_IIb", "E_com")
    amostras = {n: {k: [] for k in chaves} for n in C}
    amostras_T = []
    if nmc:
        rng = np.random.default_rng(semente)
        meio = len(ib) // 2
        for _ in range(int(nmc)):
            Ck = {n: C[n] + rng.normal(0.0, S[n]) for n in C}
            kk = int(np.argmax(np.where(r["bom"], Ck[r["banda"]], -np.inf)))
            ibk = np.arange(max(kk - meio, 0), min(kk - meio + len(ib), td.size))
            ibk = ibk[r["bom"][ibk]]
            if ibk.size == 0:
                continue
            try:
                ok = _passe_bandas(mod, Ck, S, td, ibk, t_iia)
            except ValueError:
                continue
            amostras_T.append(ok["T_pico_sed"])
            for n in C:
                for k in chaves:
                    amostras[n][k].append(ok[n][k])

    def pct(v):
        v = np.asarray(v, float); v = v[np.isfinite(v)]
        return np.percentile(v, [16, 50, 84]) if v.size > 10 else np.full(3, np.nan)

    out["amostras"] = {n: {k: np.asarray(v, float) for k, v in d.items()}
                       for n, d in amostras.items()}
    out["amostras_T"] = np.asarray(amostras_T, float)
    out["mc"] = {n: {k: pct(amostras[n][k]) for k in chaves} for n in C}
    out["mc"]["T_pico_sed"] = pct(amostras_T)
    return out


# ── tabela ──────────────────────────────────────────────────────────────────
def tabela_por_banda(r, pb=None):
    """Uma linha por banda: amplitude, área, T máximo e energia nas variantes."""
    pb = pb or por_banda(r)
    linhas = []
    for n in _ordem(r):
        o, m = pb[n], pb["mc"][n]
        linhas.append({
            "banda": n,
            "λ_ef (nm)": modelo.bandas[n].lambda_efetivo() * 1e9,
            "amp (ppt)": o["dC_pico"] * 1e3, "S/R": o["snr"],
            "A IIa (cm²)": o["A_IIa"], "A IIb (cm²)": o["A_IIb"],
            "% do disco": o["A_IIb"] / modelo.A_star * 100,
            "T máx IIb (K)": np.nanmax(o["T_IIb"]),
            "T máx área comum (K)": np.nanmax(o["T_com"]),
            "L pico (erg/s)": np.nanmax(o["L_IIb"]),
            "E IIa (erg)": o["E_IIa"], "σE IIa": 0.5 * (m["E_IIa"][2] - m["E_IIa"][0]),
            "E IIb (erg)": o["E_IIb"], "σE IIb": 0.5 * (m["E_IIb"][2] - m["E_IIb"][0]),
            "E área comum (erg)": o["E_com"]})
    tab = pd.DataFrame(linhas)
    ref = tab.loc[tab["banda"] == r["banda"], "E IIb (erg)"]
    if len(ref) and np.isfinite(ref.iloc[0]) and ref.iloc[0] > 0:
        tab[f"E / E({r['banda']})"] = tab["E IIb (erg)"] / ref.iloc[0]
    return tab


# ── figura 1: uma COLUNA por banda, uma LINHA por grandeza ──────────────────
def plota_detalhe_por_banda(r, pb=None, figsize=None):
    """Grade banda × grandeza: curva, resíduo, T(t), L(t) e energia acumulada.

    Cada coluna é uma banda; cada linha compartilha o eixo y, então a comparação
    entre bandas é direta. O modelo desenhado sobre os pontos (linha 1) é o da
    solução da banda de DETECÇÃO — nas outras três ele é previsão, não ajuste,
    e é por isso que a linha 2 (resíduo) tem conteúdo.
    """
    pb = pb or por_banda(r)
    nomes = _ordem(r)
    tm = r["td"] * 1440
    tpico = tm[r["ibins_pico"]]
    figsize = figsize or (4.1 * len(nomes), 13)
    fig, ax = plt.subplots(5, len(nomes), figsize=figsize, sharex="col", sharey="row")
    ax = np.atleast_2d(ax)

    T_max_glob = max(np.nanmax(pb[n]["T_IIb"]) for n in nomes)

    for j, n in enumerate(nomes):
        o = pb[n]
        cor = CORES[n]
        # previsão da solução global (a da banda de detecção) nesta banda
        mod_glob = _prev(modelo, n, r["A_IIb"], r["T_IIb"])
        res = (r["C"][n] - 1.0 - mod_glob) / r["S"][n]
        chi2red = float(np.nansum(res[r["bom"]] ** 2) / max(r["bom"].sum() - 1, 1))

        # (1) curva de luz e modelo
        a = ax[0, j]
        a.axvspan(tpico[0], tpico[-1], color="0.6", alpha=.15)
        a.errorbar(tm, (r["C"][n] - 1) * 1e3, r["S"][n] * 1e3, fmt=".", ms=4,
                   lw=.6, color=cor, alpha=.75, label="dados")
        a.plot(tm, mod_glob * 1e3, "-", lw=1.7, color="k", alpha=.8,
               label=f"IIb global ({r['banda']})")
        a.plot(tm, _prev(modelo, n, o["A_IIb"], o["T_IIb"]) * 1e3, "--", lw=1.2,
               color="crimson", label="IIb desta banda")
        a.axhline(0, color="k", lw=.6)
        a.set_title(f"{n}  ·  λ_ef = {modelo.bandas[n].lambda_efetivo()*1e9:.0f} nm\n"
                    f"amp = {o['dC_pico']*1e3:+.2f} ppt   ({o['snr']:.1f}σ)",
                    fontsize=11)
        if j == 0:
            a.set_ylabel("C − 1  [ppt]")
            a.legend(fontsize=8, loc="upper right")

        # (2) resíduo do modelo global, em sigma
        a = ax[1, j]
        a.axhspan(-1, 1, color="0.8", alpha=.5)
        a.plot(tm, res, ".", ms=4, color=cor)
        a.axhline(0, color="k", lw=.8)
        a.set_ylim(-6, 6)
        a.text(.02, .93, f"χ²red = {chi2red:.2f}", transform=a.transAxes,
               fontsize=9, va="top")
        if j == 0:
            a.set_ylabel("resíduo do IIb global [σ]")

        # (3) temperatura
        a = ax[2, j]
        a.plot(tm, r["T_MB"], ".", ms=3, color="0.65", label="MB (4 bandas)")
        a.plot(tm, o["T_IIb"], "-", lw=1.8, color=cor, label="IIb (área própria)")
        a.plot(tm, o["T_com"], "-", lw=1.2, color="crimson", alpha=.85,
               label="área comum (SED)")
        a.plot(tm, o["T_IIa"], "--", lw=1.1, color="tab:blue",
               label=f"IIa ({r['t_pico_iia']:.0f} K)")
        a.axhline(pb["T_pico_sed"], color="k", ls=":", lw=1.1)
        a.axhline(modelo.T_star, color="k", lw=.8)
        a.set_ylim(modelo.T_star * .9, 1.25 * T_max_glob)
        a.text(.02, .93, f"T máx = {np.nanmax(o['T_IIb']):.0f} K\n"
                         f"A = {o['A_IIb']:.2e} cm²",
               transform=a.transAxes, fontsize=9, va="top")
        if j == 0:
            a.set_ylabel("T da flare [K]")
            a.legend(fontsize=8, loc="upper right")

        # (4) luminosidade
        a = ax[3, j]
        a.plot(tm, o["L_IIb"], "-", lw=1.8, color=cor, label="IIb")
        a.plot(tm, o["L_IIa"], "--", lw=1.1, color="tab:blue", label="IIa")
        a.plot(tm, o["L_com"], "-", lw=1.0, color="crimson", alpha=.8,
               label="área comum")
        a.text(.02, .93, f"L pico = {np.nanmax(o['L_IIb']):.2e} erg/s",
               transform=a.transAxes, fontsize=9, va="top")
        if j == 0:
            a.set_ylabel("L da flare [erg/s]")
            a.legend(fontsize=8, loc="upper right")

        # (5) energia acumulada
        a = ax[4, j]
        for tag, est, c2 in (("IIb", "-", cor), ("IIa", "--", "tab:blue"),
                             ("com", "-", "crimson")):
            a.plot(tm, _cum(r["td"], o[f"L_{tag}"]), est,
                   lw=1.6 if tag == "IIb" else 1.1, color=c2, alpha=.9,
                   label={"com": "área comum"}.get(tag, tag))
        m = pb["mc"][n]["E_IIb"]
        if np.isfinite(m).all():
            a.fill_between(tm[[0, -1]], m[0], m[2], color=cor, alpha=.12)
        a.text(.02, .93,
               f"E IIb = {o['E_IIb']:.2e} erg" +
               (f"\n(−{o['E_IIb']-m[0]:.1e} +{m[2]-o['E_IIb']:.1e})"
                if np.isfinite(m).all() else ""),
               transform=a.transAxes, fontsize=9, va="top")
        a.set_xlabel("minutos desde o início da janela")
        if j == 0:
            a.set_ylabel("energia acumulada [erg]")
            a.legend(fontsize=8, loc="lower right")

    fig.suptitle(f"{r['id']} · {rotulo_noite(r['noite'])} · cada banda como se "
                 f"fosse a única  |  T pico SED = {pb['T_pico_sed']:.0f} K, "
                 f"área da SED = {pb['A_sed']:.2e} cm²", y=.998, fontsize=12)
    fig.tight_layout()
    return fig


# ── figura 2: comparação entre bandas ───────────────────────────────────────
def plota_comparacao_bandas(r, pb=None, figsize=(14, 9)):
    """Área, energia, SED do pico e a distribuição Monte Carlo, banda a banda."""
    pb = pb or por_banda(r)
    nomes = _ordem(r)
    x = np.arange(len(nomes))
    fig, ax = plt.subplots(2, 2, figsize=figsize)
    a, b, c, e = ax[0, 0], ax[0, 1], ax[1, 0], ax[1, 1]

    def barras(eixo, chave, deslocs):
        for tag, dx, mk, cor in deslocs:
            y = np.array([pb[n][f"{chave}_{tag}"] for n in nomes])
            er = np.array([[max(pb[n][f"{chave}_{tag}"] - pb["mc"][n][f"{chave}_{tag}"][0], 0),
                            max(pb["mc"][n][f"{chave}_{tag}"][2] - pb[n][f"{chave}_{tag}"], 0)]
                           if f"{chave}_{tag}" in pb["mc"][n] else [0., 0.]
                           for n in nomes]).T
            er = np.where(np.isfinite(er), er, 0.0)
            eixo.errorbar(x + dx, y, er, fmt=mk, ms=8, color=cor, capsize=3,
                          label={"com": "área comum"}.get(tag, tag))

    # (a) áreas
    barras(a, "A", [("IIa", -.09, "o", "tab:blue"), ("IIb", .09, "s", "crimson")])
    a.axhline(pb["A_sed"], color="k", ls="--", lw=1.2, label="área da SED")
    a.set_xticks(x); a.set_xticklabels(nomes); a.set_yscale("log")
    a.set_ylabel("área deprojetada [cm²]"); a.legend(fontsize=9)
    a.set_title("área da Eq. (6), banda a banda")

    # (b) energias
    barras(b, "E", [("IIa", -.09, "o", "tab:blue"), ("IIb", .09, "s", "crimson"),
                    ("com", 0.0, "^", "0.45")])
    if np.isfinite(r.get("E_IIb", np.nan)):
        b.axhline(r["E_IIb"], color="k", ls="--", lw=1.2,
                  label=f"medida da célula 10 ({r['banda']})")
    b.set_xticks(x); b.set_xticklabels(nomes); b.set_yscale("log")
    b.set_ylabel("energia bolométrica [erg]"); b.legend(fontsize=9)
    b.set_title("energia, banda a banda")

    # (c) SED do pico
    lam = np.array([modelo.bandas[n].lambda_efetivo() for n in nomes]) * 1e9
    dC = np.array([pb[n]["dC_pico"] for n in nomes]) * 1e3
    sg = np.array([pb[n]["sig_pico"] for n in nomes]) * 1e3
    c.errorbar(lam, dC, sg, fmt="o", color="k", ms=8, label="medido no pico")
    for n in nomes:
        c.plot(modelo.bandas[n].lambda_efetivo() * 1e9,
               pb[n]["dC_pico"] * 1e3, "o", ms=8, color=CORES[n])
    for T, A, cor, est, rot in (
            (pb["T_pico_sed"], pb["A_sed"], "crimson", "-", "SED (4 bandas)"),
            (r["t_pico_iia"], modelo.area_pico(pb[r["banda"]]["dC_pico"],
                                               r["t_pico_iia"], r["banda"])[1],
             "tab:blue", "--", f"IIa {r['t_pico_iia']:.0f} K")):
        if not np.isfinite(T) or T <= modelo.T_star:
            continue
        prev = [A * modelo.mu / modelo.A_star *
                (modelo.bandas[n].I(T) / modelo.I_star[n] - 1) * 1e3 for n in nomes]
        c.plot(lam, prev, est, color=cor, lw=1.6, label=f"corpo negro {T:.0f} K · {rot}")
    c.set_xlabel("λ efetivo da banda [nm]"); c.set_ylabel("C − 1 no pico [ppt]")
    c.set_yscale("log"); c.legend(fontsize=9); c.set_title("SED do pico")

    # (d) distribuição Monte Carlo da energia IIb
    for n in nomes:
        v = pb["amostras"].get(n, {}).get("E_IIb", np.array([]))
        v = v[np.isfinite(v)]
        if v.size > 10:
            e.hist(v, bins=30, histtype="step", lw=1.6, color=CORES[n], label=n)
    e.axvline(r.get("E_IIb", np.nan), color="k", ls="--", lw=1.2,
              label=f"medida ({r['banda']})")
    e.set_xlabel("energia IIb [erg]"); e.set_ylabel("realizações")
    e.legend(fontsize=9)
    e.set_title(f"Monte Carlo ({pb['amostras_T'].size} realizações)")

    fig.suptitle(f"{r['id']} · comparação entre bandas", y=1.0, fontsize=12)
    fig.tight_layout()
    return fig


# ── execução ────────────────────────────────────────────────────────────────
POR_BANDA, TAB_BANDA = {}, {}
FMT_B = {"λ_ef (nm)": "{:.0f}", "amp (ppt)": "{:+.2f}", "S/R": "{:.1f}",
         "A IIa (cm²)": "{:.2e}", "A IIb (cm²)": "{:.2e}", "% do disco": "{:.3f}",
         "T máx IIb (K)": "{:.0f}", "T máx área comum (K)": "{:.0f}",
         "L pico (erg/s)": "{:.2e}", "E IIa (erg)": "{:.2e}", "σE IIa": "{:.1e}",
         "E IIb (erg)": "{:.2e}", "σE IIb": "{:.1e}", "E área comum (erg)": "{:.2e}"}

for fid, r in resultados.items():
    pb = por_banda(r)
    tab = tabela_por_banda(r, pb)
    POR_BANDA[fid], TAB_BANDA[fid] = pb, tab
    print(f"── {fid} · banda a banda " + "─" * 44)
    fmt = dict(FMT_B) | {c: "{:.2f}" for c in tab.columns if c.startswith("E / E(")}
    display(tab.style.format(fmt, na_rep="—").hide(axis="index"))
    Ei = tab["E IIb (erg)"].to_numpy(float); Ei = Ei[np.isfinite(Ei) & (Ei > 0)]
    if Ei.size > 1:
        print(f"   espalhamento da energia entre as bandas: "
              f"{(Ei.max()/Ei.min()-1)*100:.0f}%  — é a sistemática de banda "
              f"única, e a barra Monte Carlo NÃO a contém.")
    tab.to_csv(f"{PREFIXO}_{fid}_bandas.csv", index=False, float_format="%.6g")
    plota_detalhe_por_banda(r, pb); plt.show()
    plota_comparacao_bandas(r, pb); plt.show()
    print()

── 20260815_F01 · banda a banda ────────────────────────────────────────────


banda,λ_ef (nm),amp (ppt),S/R,A IIa (cm²),A IIb (cm²),% do disco,T máx IIb (K),T máx área comum (K),L pico (erg/s),E IIa (erg),σE IIa,E IIb (erg),σE IIb,E área comum (erg),E / E(g)
g,457,+177.31,62.3,6.82e+18,4.87e+18,0.057,11174,11158,4.25e+30,1.03e+33,1.4e+31,1.05e+33,3.2e+31,1.05e+33,1.00
r,614,+55.58,17.8,7.82e+18,6.00e+18,0.070,11569,12634,6.03e+30,1.15e+33,4.0e+31,1.28e+33,1.4e+32,1.42e+33,1.22
i,753,+13.19,4.7,3.76e+18,2.99e+18,0.035,14863,11549,8.23e+30,8.13e+32,2.6e+32,9.80e+32,5.3e+32,6.83e+32,0.93
z,894,+7.36,2.5,3.40e+18,2.75e+18,0.032,18941,13329,2.00e+31,1.53e+33,2.1e+33,1.99e+33,3.5e+33,1.04e+33,1.89


   espalhamento da energia entre as bandas: 103%  — é a sistemática de banda única, e a barra Monte Carlo NÃO a contém.



In [18]:
# ═══ 5b · verificação passo a passo — SÓ NA BANDA g ═════════════════════════
# Antes de rodar em dado real, cada equação do Heinzel+2026 é exercitada
# isoladamente num flare SINTÉTICO da banda g, cuja resposta verdadeira é
# conhecida. Se um passo falhar aqui, o número da célula 10 não vale nada.
#
#   Eq. (2)-(4)  C(t) construído da definição de luminosidade
#   Eq. (5)      C -> A_flare  (ida e volta exata)
#   Eq. (6)      C_max, T_pico -> A fixa   + sensibilidade a T
#   Eq. (7)      A fixa -> T(t)            (recupera o T verdadeiro)
#   Eq. (8)-(9)  L(t), E                   (caso analítico de T e A constantes)
#   Tabela 1     áreas de TIC 325178532 reproduzidas com uma banda tipo TESS
#   Figura 1     E vs T_pico assumido: o mínimo largo

VERIF_BANDA   = "g"
VERIF_T_PICO  = 11000.0     # T de pico do flare sintético
VERIF_A_FRAC  = 5.0e-4      # A_flare / A_star verdadeiro (fixo no evento)
VERIF_DUR_MIN = 60.0        # janela, em minutos
VERIF_TOL     = 1e-6        # tolerância dos testes de ida-e-volta

_b   = modelo.bandas[VERIF_BANDA]
_Is  = modelo.I_star[VERIF_BANDA]
_As  = modelo.A_star
_pass, _falha = [], []


def _ok(rot, val, ref, tol=VERIF_TOL, unidade=""):
    """Compara e registra; devolve True/False."""
    ref = float(ref); val = float(val)
    rel = abs(val - ref) / abs(ref) if ref != 0 else abs(val)
    bom = rel <= tol
    (_pass if bom else _falha).append(rot)
    print(f"   {'✓' if bom else '✗'}  {rot}: {val:.6g}{unidade} "
          f"vs {ref:.6g}{unidade}   (dif {rel*100:.3g}%, tol {tol*100:g}%)")
    return bom


print("═" * 78)
print(f"VERIFICAÇÃO PASSO A PASSO · banda {VERIF_BANDA} · "
      f"T* = {modelo.T_star:.0f} K, R* = {modelo.R_star} R_sol, A* = {_As:.4e} cm²")
print("═" * 78)

# ── passo 0 · a função resposta e as integrais de Planck ────────────────────
print("\n[0] função resposta e I(T) = ∫ S_λ B_λ(T) dλ")
print(f"   banda {VERIF_BANDA}: {_b.lam.min()*1e9:.1f}–{_b.lam.max()*1e9:.1f} nm, "
      f"λ_ef = {_b.lambda_efetivo()*1e9:.1f} nm, {_b.lam.size} pontos")
print(f"   I(T*) = {_Is:.6e} | I({VERIF_T_PICO:.0f} K) = {_b.I(VERIF_T_PICO):.6e} "
      f"| razão = {_b.I(VERIF_T_PICO)/_Is:.2f}")
# a tabela I(T) tem que ser monotônica, senão a inversão da Eq. (7) é ambígua
_ok("I(T) monotônica", float(np.all(np.diff(_b.I_grid) > 0)), 1.0, 0)
_Tt = np.array([4000., 6000., 8000., 12000., 20000.])
_ok("inversão T→I→T (máx. erro)", np.max(np.abs(_b.T_de_I(_b.I(_Tt)) - _Tt) / _Tt),
    0.0, 1e-4)
# I(T) não pode depender da normalização de S_λ: só a razão entra nas Eqs. 6-7
_b2 = Banda(VERIF_BANDA + "×3", _b.lam, _b.S * 3.0)
_ok("razão I_flare/I_star invariante a 3·S_λ",
    _b2.I(VERIF_T_PICO) / _b2.I(modelo.T_star), _b.I(VERIF_T_PICO) / _Is, 1e-9)

# ── passo 1 · Eqs. (2)-(4): construir C(t) da definição ─────────────────────
print("\n[1] Eqs. (2)-(4) · curva de luz normalizada, montada da definição")
t_min = np.arange(0.0, VERIF_DUR_MIN, BIN_S / 60.0)
td    = t_min / 1440.0
subida, descida = 4.0, 9.0
perfil = np.where(t_min < subida, t_min / subida,
                  np.exp(-(t_min - subida) / descida))
T_true = modelo.T_star + (VERIF_T_PICO - modelo.T_star) * perfil
A_true = VERIF_A_FRAC * _As                      # área FIXA, como no método IIa

F_star  = _Is                                    # ∫S_λ F_star dλ (Planck em T*)
F_flare = _b.I(T_true)                           # ∫S_λ B_λ(T_flare(t)) dλ
L       = (_As - A_true) * F_star + A_true * F_flare        # Eq. (2)
L_star  = _As * F_star                                      # Eq. (3)
C_g     = L / L_star                                        # Eq. (4)

C_compacto = 1.0 + A_true / _As * (F_flare / F_star - 1.0)
_ok("Eq. (4) explícita == forma compacta (máx.)",
    np.max(np.abs(C_g - C_compacto)), 0.0, 1e-12)
print(f"   amplitude de pico: {(C_g.max()-1)*1e3:.3f} ppt "
      f"| A_flare/A_star = {VERIF_A_FRAC:.2e}")

# ── passo 2 · Eq. (5): C -> A_flare, ponto a ponto ──────────────────────────
print("\n[2] Eq. (5) · A_flare = (C−1) A* F*/(F_flare − F*), ida e volta")
sel = perfil > 1e-3                               # onde há flare de verdade
A_eq5 = (C_g[sel] - 1.0) * _As * F_star / (F_flare[sel] - F_star)
_ok("A recuperada == A verdadeira (máx.)",
    np.max(np.abs(A_eq5 - A_true) / A_true), 0.0, 1e-9)

# ── passo 3 · Eq. (6): amplitude de pico + T assumido -> A fixa ─────────────
print("\n[3] Eq. (6) · área fixa da amplitude de pico")
dC_max = float(C_g.max() - 1.0)
A_proj, A_dep = modelo.area_pico(dC_max, VERIF_T_PICO, VERIF_BANDA)
_ok("A(Eq. 6) com o T verdadeiro == A verdadeira", A_proj, A_true, 1e-9, " cm²")
_ok("deprojeção A/μ", A_dep, A_proj / modelo.mu, 1e-12, " cm²")
# sensibilidade da área ao T assumido. O TEXTO do artigo diz ~5% por 1000 K e
# ~10% por 2000 K; a Tabela 1 do próprio artigo diz outra coisa — 4439 ppm a
# 10 000 K e 1873 ppm a 15 100 K são um fator 2.37 em 5100 K, ou seja ~17% por
# 1000 K. Aqui a conta é refeita nas duas bandas para ver quem está certo.
print("   sensibilidade da área ao T de pico assumido:")
for rot, bb, Ts, Tp in ((f"banda {VERIF_BANDA} (T* = {modelo.T_star:.0f} K)",
                         _b, modelo.T_star, VERIF_T_PICO),
                        ("TESS top-hat (T* = 4000 K, o caso da Tabela 1)",
                         Banda.tophat("TESS", 600e-9, 1000e-9, 3000), 4000.0, 10000.0)):
    Is_b = float(bb.I(Ts)); base = 1.0 / (bb.I(Tp) - Is_b)
    d1  = (1.0 / (bb.I(Tp + 1000.) - Is_b) / base - 1) * 100
    d2v = (1.0 / (bb.I(Tp + 2000.) - Is_b) / base - 1) * 100
    print(f"      {rot}, T_pico = {Tp:.0f} K: "
          f"ΔT=+1000 K -> {d1:+.1f}% | ΔT=+2000 K -> {d2v:+.1f}%")
print("      o texto do artigo promete ~5% e ~10%; a Tabela 1 dele implica ~17%/1000 K")
print("      -> a área NÃO é insensível ao T assumido, e em g menos ainda")

# ── passo 4 · Eq. (7): A fixa -> T(t) ──────────────────────────────────────
print("\n[4] Eq. (7) · temperatura instante a instante, com a área fixa")
T_rec = modelo.temperatura(C_g, A_proj, VERIF_BANDA)
_ok("T recuperado == T verdadeiro, onde há flare (máx.)",
    np.max(np.abs(T_rec[sel] - T_true[sel]) / T_true[sel]), 0.0, 1e-4)
_ok("T no pico == T assumido", T_rec.max(), VERIF_T_PICO, 1e-4, " K")
C_rec = 1.0 + A_proj / _As * (_b.I(T_rec) / _Is - 1.0)
_ok("C reconstruído == C original (máx.)", np.max(np.abs(C_rec - C_g)), 0.0, 1e-9)

# ── passo 5 · Eqs. (8)-(9): luminosidade e energia ──────────────────────────
print("\n[5] Eqs. (8)-(9) · L(t) = σ T⁴ A  e  E = ∫ L dt")
E_rec, L_rec = modelo.energia(td, T_rec, A_dep)
E_true, _    = modelo.energia(td, T_true, A_dep)
_ok("E do T recuperado == E do T verdadeiro", E_rec, E_true, 1e-4, " erg")
T_c = np.full(td.size, 9000.0)
E_c, _ = modelo.energia(td, T_c, A_dep, subtrair_piso=False)
_ok("caso T,A constantes == σT⁴AΔt",
    E_c, SIGMA_SB * 9000.0 ** 4 * A_dep * (td[-1] - td[0]) * 86400.0, 1e-9, " erg")
E_sem, _ = modelo.energia(td, T_rec, A_dep, subtrair_piso=False)
print(f"   piso σT*⁴A ao longo da janela: {E_sem - E_rec:.3e} erg "
      f"({(E_sem/E_rec - 1)*100:.1f}% de E) — cresce linearmente com a janela")
print(f"   E = {E_rec:.4e} erg | L de pico = {L_rec.max():.4e} erg/s "
      f"| duração equivalente = {E_rec/L_rec.max()/60:.1f} min")

# ── passo 6 · Tabela 1 do artigo: TIC 325178532 (banda tipo TESS) ───────────
print("\n[6] Tabela 1 do artigo · TIC 325178532, K7V "
      "(banda TESS aproximada por 600–1000 nm)")
tess = Banda.tophat("TESS", 600e-9, 1000e-9, 3000)
mod_tic = FlareHeinzel(4000.0, 0.9, [tess], mu=1.0)
dC_tic  = 0.08679
for rot, T_p, ref in (("Método I  (sem o −F*)", 10000.0, 4223.0),
                      ("Método IIa (T = 10 000 K)", 10000.0, 4439.0),
                      ("Método IIb (T = 15 100 K)", 15100.0, 1873.0)):
    Is_t = mod_tic.I_star["TESS"]
    den = (tess.I(T_p) if rot.startswith("Método I ") else tess.I(T_p) - Is_t)
    ppm = dC_tic * Is_t / den * 1e6
    _ok(f"{rot} · área [ppm]", ppm, ref, 0.05, " ppm")
print("   (a diferença de poucos % é a resposta TESS aproximada por top-hat;\n"
      "    o que se verifica aqui é a ÁLGEBRA das Eqs. 5-6, não o filtro)")

# ── passo 7 · Figura 1 do artigo: E vs T de pico assumido ──────────────────
print("\n[7] Figura 1 do artigo · energia vs T de pico assumido")
T_ass = np.arange(modelo.T_star + 300.0, 20001.0, 250.0)
E_art, E_nos = [], []
for T_p in T_ass:
    A_p, A_d = modelo.area_pico(dC_max, T_p, VERIF_BANDA)
    T_p_t = modelo.temperatura(C_g, A_p, VERIF_BANDA)
    E_art.append(modelo.energia(td, T_p_t, A_d, subtrair_piso=False)[0])
    E_nos.append(modelo.energia(td, T_p_t, A_d, subtrair_piso=True)[0])
E_art, E_nos = np.array(E_art), np.array(E_nos)
k_art, k_nos = int(np.argmin(E_art)), int(np.argmin(E_nos))
print(f"   como no artigo (sem subtrair o piso): mínimo em {T_ass[k_art]:.0f} K, "
      f"E(4000)/E(6000) = {np.interp(4000., T_ass, E_art)/np.interp(6000., T_ass, E_art):.1f}"
      f"   (o artigo cita ~8 para EV Lac)")
print(f"   com o piso σT*⁴ subtraído:           mínimo em {T_ass[k_nos]:.0f} K, "
      f"E(4000)/E(6000) = {np.interp(4000., T_ass, E_nos)/np.interp(6000., T_ass, E_nos):.1f}")
print("   -> o ramo frio da Figura 1 do artigo é, em boa parte, a energia que a\n"
      "      fotosfera emitiria de qualquer jeito naquela área; subtraí-la mata a subida")
_ok("mínimo interior (não numa borda), versão do artigo",
    float(0 < k_art < E_art.size - 1), 1.0, 0)

# ── figuras ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
a, b, c, d2, e, f = ax.ravel()
cor = CORES.get(VERIF_BANDA, "tab:blue")

# (a) S_λ e o contraste flare/estrela dentro da banda
lam_nm = _b.lam * 1e9
a.fill_between(lam_nm, _b.S, color=cor, alpha=.25, label=f"S_λ ({VERIF_BANDA})")
a.plot(lam_nm, _b.S, color=cor, lw=1.4)
a.axvline(_b.lambda_efetivo() * 1e9, color="k", ls=":", lw=1.1, label="λ_ef")
a.set_xlabel("λ [nm]"); a.set_ylabel("transmissão"); a.legend(fontsize=9, loc="upper left")
a2 = a.twinx()
r_lam = (_planck(_b.lam, np.array([VERIF_T_PICO]))[0]
         / _planck(_b.lam, np.array([modelo.T_star]))[0])
a2.plot(lam_nm, r_lam, "-", color="crimson", lw=1.6)
a2.set_ylabel("B_λ(T_flare)/B_λ(T*)", color="crimson"); a2.set_yscale("log")
a.set_title("[0] resposta e contraste dentro da banda")

# (b) C(t): construído x reconstruído
b.plot(t_min, (C_g - 1) * 1e3, "-", lw=2.4, color=cor, label="Eqs. (2)-(4)")
b.plot(t_min, (C_rec - 1) * 1e3, "--", lw=1.3, color="k", label="reconstruído da Eq. (7)")
b.set_xlabel("minutos"); b.set_ylabel("C − 1 [ppt]"); b.legend(fontsize=9)
b.set_title(f"[1,4] curva sintética · pico {dC_max*1e3:.1f} ppt")

# (c) área da Eq. (6) vs T assumido
A_ass = np.array([modelo.area_pico(dC_max, T, VERIF_BANDA)[0] for T in T_ass])
c.plot(T_ass, A_ass / A_true, "-", lw=1.8, color=cor)
c.axhline(1.0, color="k", lw=.8); c.axvline(VERIF_T_PICO, color="k", ls=":", lw=1.1)
for dT, cor_dt in ((1000.0, "0.5"), (2000.0, "0.75")):
    for s in (+1, -1):
        c.axvline(VERIF_T_PICO + s * dT, color=cor_dt, ls="--", lw=.9)
c.set_yscale("log"); c.set_xlabel("T de pico assumido [K]")
c.set_ylabel("A(Eq. 6) / A verdadeira")
c.set_title("[3] a área depende do T assumido")

# (d) T(t): verdadeiro x recuperado, e o resíduo
d2.plot(t_min, T_true, "-", lw=2.4, color="0.6", label="verdadeiro")
d2.plot(t_min, T_rec, "--", lw=1.4, color=cor, label="Eq. (7)")
d2.axhline(modelo.T_star, color="k", lw=.8)
d2.set_xlabel("minutos"); d2.set_ylabel("T [K]"); d2.legend(fontsize=9, loc="upper right")
d3 = d2.twinx()
d3.plot(t_min[sel], (T_rec[sel] - T_true[sel]), ".", ms=3, color="crimson")
d3.set_ylabel("T_rec − T_true [K]", color="crimson")
d2.set_title("[4] temperatura recuperada")

# (e) L(t) e energia acumulada
e.plot(t_min, L_rec, "-", lw=1.8, color=cor)
e.set_xlabel("minutos"); e.set_ylabel("L [erg/s]", color=cor)
e2 = e.twinx()
_ts = td * 86400.0
_Ecum = np.concatenate([[0.0], np.cumsum(0.5 * (L_rec[1:] + L_rec[:-1]) * np.diff(_ts))])
e2.plot(t_min, _Ecum, "-", lw=1.6, color="crimson")
e2.set_ylabel("E acumulada [erg]", color="crimson")
e.set_title(f"[5] E = {E_rec:.3e} erg")

# (f) reprodução da Figura 1 do artigo
f.plot(T_ass, E_art / 1e33, "-", lw=1.8, color="crimson", label="como no artigo")
f.plot(T_ass, E_nos / 1e33, "--", lw=1.6, color=cor, label="piso σT*⁴ subtraído")
f.plot(T_ass[k_art], E_art[k_art] / 1e33, "o", ms=8, color="crimson")
f.plot(T_ass[k_nos], E_nos[k_nos] / 1e33, "o", ms=8, color=cor)
f.axvline(VERIF_T_PICO, color="k", ls=":", lw=1.2, label="T verdadeiro")
f.set_yscale("log")
f.set_xlabel("T máximo assumido do flare [K]"); f.set_ylabel("energia [10³³ erg]")
f.legend(fontsize=9); f.set_title("[7] Figura 1 do artigo, reproduzida")

fig.suptitle(f"verificação das Eqs. (2)-(9) do Heinzel+2026 · banda "
             f"{VERIF_BANDA} · flare sintético", y=1.0, fontsize=13)
fig.tight_layout()
plt.show()

print("\n" + "═" * 78)
print(f"{len(_pass)} testes passaram, {len(_falha)} falharam"
      + (": " + ", ".join(_falha) if _falha else ""))
print("═" * 78)

NameError: name 'modelo' is not defined

In [2]:
from synphot import SourceSpectrum
from synphot.models import BlackBodyNorm1D

# Approximate using a blackbody model at ~3700 K
au_mic_spec = SourceSpectrum(BlackBodyNorm1D, temperature=3700)


In [ ]:
caminho_curvas = os.path.join(OUTDIR, "aumic_flares_20260815_F01_curvas.csv")
dados = pd.read_csv(caminho_curvas)

tt = dados["minutos"].to_numpy(float)       # 77 tempos: 0 a 19 min
C_menos_1 = dados["C_g"].to_numpy(float) - 1.0
mu = MU

FileNotFoundError: [Errno 2] No such file or directory: 'Sparc4-data/resultados_flares/aumic_flares_20260815_F01_curvas.csv'